In [1]:
# ==========================================
# CELL 0: Setup & Imports
# ==========================================
!pip install mne isotree

import pathlib as pl
import numpy as np
import mne
import pywt
import pandas as pd
from isotree import IsolationForest
from scipy.spatial.distance import jensenshannon
import matplotlib.pyplot as plt

from google.colab import drive
drive.mount('/content/drive')

print("Libraries imported and Google Drive mounted successfully.")

Mounted at /content/drive
Libraries imported and Google Drive mounted successfully.


In [2]:
# ==========================================
# CELL 1: Dataset Loading
# ==========================================
import pathlib as pl
import mne

def loadDataset(name, subject, montageName, usedChannels, rootPath, montageChannelNames, parametersOfDenoiser=None, verbose=False):
    """
    Loads the EEG dataset according to the pre-defined folder structure.
    Extracts the shifted data version for temporal alignment.
    """
    dataset = pl.Path(name)
    subject = str(subject)
    root = pl.Path(rootPath)

    # Construct the file pathway for raw EEG data files
    pathDataset = pl.Path.joinpath(root, dataset.name + " All Data and Scripts/")
    filenameSubject = pl.Path(subject + "_" + dataset.name + "_" + "shifted.set")
    pathDatasetSubject = pl.Path.joinpath(pathDataset, subject, filenameSubject)

    # Construct the file pathway for the EEG sensor coordinate (Montage) file
    filenameMontage = pl.Path(str("standard-10-5-cap385.elp"))
    montagePath = pl.Path.joinpath(root, dataset.name + " All Data and Scripts/", "EEG_ERP_Processing", filenameMontage)

    # Ingest EEGLAB format files using the MNE framework
    dataForSubject = mne.io.read_raw_eeglab(pathDatasetSubject, preload=True)
    dataForSubject.rename_channels(lambda s: s.strip("."))

    channels = dataForSubject.info["ch_names"]
    sampleFreq = dataForSubject.info["sfreq"]

    if verbose:
        print("Data summary before pre-processing: {}".format(dataForSubject))
        print("Metadata structure before pre-processing: {}".format(dataForSubject.info))

        print("Loading file: " + str(filenameSubject))
        print("Root path: ", root)
        print("Dataset directory: ", pathDataset)
        print("Subject path location: ", pathDatasetSubject)
        print("Montage file path: ", montagePath)
        print("Data successfully loaded for subject: {}".format(subject))
        print("Identified channels: {}".format(channels))

        # Output experimental event markings and stimulus annotations
        print("Annotations data: " + str(dataForSubject.annotations))
        print("Annotation durations shape: " + str(dataForSubject.annotations.duration.shape))
        print("Annotation descriptions shape: " + str(dataForSubject.annotations.description.shape))
        print("Annotation onsets shape: " + str(dataForSubject.annotations.onset.shape))
        if len(dataForSubject.annotations.onset) > 0:
            print("Initial event timestamp: " + str(dataForSubject.annotations.onset[0]))
        print(list(dataForSubject.annotations.description))

        # Render raw EEG signal configurations
        dataForSubject.plot()

    return dataForSubject

In [3]:
# ==========================================
# CELL 2: Preprocessing Utilities
# ==========================================
import numpy as np
import mne

#Channel definitions for standard 10-20 montage
EEG_30 = ["FP1","F3","F7","FC3","C3","C5","P3","P7","P9","PO7","PO3","O1","Oz","Pz","CPz",
          "FP2","Fz","F4","F8","FC4","FCz","Cz","C4","C6","P4","P8","P10","PO8","PO4","O2"]
EOG_CH = ["HEOG_left","HEOG_right","VEOG_lower"]

def fix_channel_types(raw):
    """
    Relabel EOG channels from 'eeg' to 'eog' type to prevent contamination during average referencing.
    """
    r = raw.copy()
    present = {c:'eog' for c in EOG_CH if c in r.ch_names}
    if present:
        r.set_channel_types(present, verbose='ERROR')
    return r

def preprocess_avg_reref(raw):
    """
    Apply 1Hz highpass filter and average reference to EEG channels.
    Adds temporary flat channel to stabilize reference calculation.
    Returns 30 EEG channels.
    """
    r = fix_channel_types(raw).load_data()
    r.pick(EEG_30)
    r.filter(1., None, verbose='ERROR')

    #Add flat channel for stable average reference
    flat = mne.io.RawArray(np.zeros((1, r.n_times)),
                           mne.create_info(['FLAT'], r.info['sfreq'], ['eeg']),
                           verbose='ERROR')
    r.add_channels([flat], force_update_info=True)
    r.set_eeg_reference('average', verbose='ERROR')
    r.drop_channels(['FLAT'])

    return r

In [4]:
# ==========================================
# CELL 3: EEG Windowing
# ==========================================
class OnEEGWaveLAD_Windowing:
    def __init__(self, raw_data, RTWL=1000, clean_data=None, noisy_data=None):
        self.raw = raw_data
        self.Sr = raw_data.info['sfreq']
        self.RTWL = RTWL
        theo = (self.RTWL/1000.0)*self.Sr
        self.window_samples = int(2**np.ceil(np.log2(theo)))
        self.actual_RTWL = (self.window_samples/self.Sr)*1000.0
        print(f"[Phase A] window={self.actual_RTWL:.2f} ms ({self.window_samples} samples).")
        self.total_samples = raw_data.n_times
        self._clean = clean_data     #Reference clean signal (e.g., ICA-denoised); None=use raw
        self._noisy = noisy_data     #Contaminated signal with injected artifacts; None=fallback behavior

    def get_window_stream(self):
        for s in range(0, self.total_samples, self.window_samples):
            e = s + self.window_samples
            if e > self.total_samples:
                break
            if self._clean is not None:
                curr_clean = self._clean[:, s:e]
                curr_noisy = self._noisy[:, s:e] if self._noisy is not None else curr_clean
                yield {"original_window": curr_clean,   #Clean ground truth
                       "dwt_input": curr_noisy}         #Clean + known artifacts
            else:
                yield {"original_window": self.raw.get_data()[:, s:e],
                       "dwt_input": self.raw.get_data()[:, s:e]}

In [5]:
# ==========================================
# CELL 4: Multi-level Discrete Wavelet Decomposition
# ==========================================
class OnEEGWaveLAD_DWT:
    def __init__(self, MW='sym4', decomposition_level=None):
        self.MW = MW
        self.level = decomposition_level

    def decompose_window(self, dwt_input_data):
        signal_length = dwt_input_data.shape[1]

        if self.level is None:
            # Determine the maximum mathematically valid decomposition level for the selected mother wavelet
            max_level = pywt.dwt_max_level(signal_length, self.MW)
            # Select the optimal decomposition level constrained by the signal boundary conditions
            actual_level = min(int(np.log2(signal_length)), max_level)
        else:
            actual_level = self.level

        return [pywt.wavedec(dwt_input_data[ch], self.MW, level=actual_level, mode='periodization')
                for ch in range(dwt_input_data.shape[0])]

In [6]:
# ==========================================
# CELL 5: Multi-scale Anomaly Detection and Signal Reconstruction
# ==========================================
from isotree import IsolationForest
import numpy as np
import pywt

class OnEEGWaveLAD_Denoiser:
    """
    Core Unsupervised Denoising Framework.
    Implements scaleogram formation, anomaly detection via Extended Isolation Forest,
    and adaptive artifact attenuation.
    """
    def __init__(self, n_channels, Bs=20, IFt=100, IFS=512, Ta=0.55, Es=35, el=None):
        self.n_channels = n_channels
        self.Bs = Bs
        self.IFt = IFt
        self.IFS = IFS
        self.Ta = Ta
        self.Es = Es
        self.el = el

        self.moving_buffers_history = [[] for _ in range(n_channels)]
        self.models = [None for _ in range(n_channels)]
        self.centroids = [None] * n_channels
        self.max_dists = [None] * n_channels

        #Counter to control retraining frequency
        self.window_cnt = 0

    def process_window(self, all_channels_coeffs, target_len, MW='sym4', return_debug=False, debug_channel=0):
        self.window_cnt += 1
        denoised_signals = []
        debug_info = {}

        n_isps = target_len // 2

        for ch in range(self.n_channels):
            coeffs = all_channels_coeffs[ch]
            cA_baseline = coeffs[0]
            details = coeffs[1:]
            num_scales = len(details)

            raw_scaleogram = np.zeros((n_isps, num_scales))
            norm_scaleogram = np.zeros((n_isps, num_scales))

            for i, c in enumerate(details):
                m_level = num_scales - i
                repeats = n_isps // len(c)
                upsampled = np.repeat(c, repeats)
                raw_scaleogram[:, i] = upsampled
                norm_scaleogram[:, i] = upsampled / (2 ** m_level)

            #Store intermediate matrices for visualization
            if return_debug and ch == debug_channel:
                debug_info['Step_C_Scaleogram_Raw'] = raw_scaleogram.copy()
                debug_info['Step_C_Scaleogram_Before'] = norm_scaleogram.copy()

            if self.models[ch] is not None:
                scores = self.models[ch].predict(norm_scaleogram)
            else:
                scores = np.zeros(n_isps)

            if return_debug and ch == debug_channel:
                debug_info['Step_E_Scores'] = scores.copy()

            self.moving_buffers_history[ch].append(norm_scaleogram)
            if len(self.moving_buffers_history[ch]) > self.Bs:
                self.moving_buffers_history[ch].pop(0)

            moving_buffer = np.vstack(self.moving_buffers_history[ch])

            if return_debug and ch == debug_channel:
                debug_info['Step_D_Moving_Buffer'] = moving_buffer.copy()

            #Use centroids instead of medoids
            self.centroids[ch] = np.mean(moving_buffer, axis=0)
            self.max_dists[ch] = np.max(np.abs(moving_buffer - self.centroids[ch]), axis=0) + 1e-8

            #Retrain every 5 windows for efficiency
            if moving_buffer.shape[0] >= self.IFS:
                if self.models[ch] is None or self.window_cnt % 5 == 0:
                    c_dim = moving_buffer.shape[1]
                    actual_ndim = c_dim if self.el is None else min(self.el + 1, c_dim)
                    self.models[ch] = IsolationForest(ntrees=self.IFt, sample_size=self.IFS, ndim=actual_ndim, random_seed=42, nthreads=-1)
                    self.models[ch].fit(moving_buffer)

            AT_exp = {idx + e for idx in np.where(scores > self.Ta)[0]
                      for e in range(-self.Es, self.Es + 1) if 0 <= idx + e < n_isps}

            mitigator = np.ones((n_isps, num_scales))

            for idx in AT_exp:
                dist_vec = np.abs(norm_scaleogram[idx] - self.centroids[ch])
                mtga_vec = 1.0 - (dist_vec / self.max_dists[ch])
                mtga_vec = np.clip(mtga_vec, 0.0, 1.0)
                mitigator[idx] = np.minimum(mitigator[idx], mtga_vec)

            if return_debug and ch == debug_channel:
                debug_info['Step_F_Mitigator'] = mitigator.copy()

            denoised_scaleogram = raw_scaleogram * mitigator

            if return_debug and ch == debug_channel:
                mod_norm_scaleogram = np.zeros_like(denoised_scaleogram)
                for i in range(num_scales):
                    m_level = num_scales - i
                    mod_norm_scaleogram[:, i] = denoised_scaleogram[:, i] / (2 ** m_level)
                debug_info['Step_C_Scaleogram_After'] = mod_norm_scaleogram.copy()

            reconstructed_coeffs = [cA_baseline.copy()]

            for i in range(num_scales):
                c = details[i]
                repeats = n_isps // len(c)
                #Aggregate mitigator by repeat groups and apply to original coefficients
                eff_mit = mitigator[:, i].reshape(len(c), repeats)[:, 0]
                reconstructed_coeffs.append(c * eff_mit)

            denoised_sig = pywt.waverec(reconstructed_coeffs, MW, mode='periodization')
            denoised_signals.append(denoised_sig[:target_len])

        if return_debug:
            return np.array(denoised_signals), debug_info
        return np.array(denoised_signals)

In [7]:
# ==========================================
# CELL 6: Pipeline Configuration & Data Ingestion
# ==========================================
import matplotlib.animation as animation
from IPython.display import Image, display
import matplotlib.pyplot as plt
import numpy as np

datasetName = "N170"
subject_id = "1"
rootPath = "/content/drive/MyDrive/EEG_Project/"

#Configuration mappings for standard channels and montages
montageName = "standard_1020"
usedChannels = ["FP1","F3","F7","FC3","C3","C5","P3","P7","P9","PO7","PO3","O1","Oz","Pz","CPz","FP2","Fz","F4","F8","FC4","FCz","Cz","C4","C6","P4","P8","P10","PO8","PO4","O2"]
montageChannelNames = ["Fp1","F3","F7","FC3","C3","C5","P3","P7","P9","PO7","PO3","O1","Oz","Pz","CPz","Fp2","Fz","F4","F8","FC4","FCz","Cz","C4","C6","P4","P8","P10","PO8","PO4","O2"]
parametersOfDenoiser = None

print("Initiating dataset ingestion and pipeline parameters mapping.")

#Read source files using structural specifications
raw_data = loadDataset(
    name=datasetName,
    subject=subject_id,
    montageName=montageName,
    usedChannels=usedChannels,
    rootPath=rootPath,
    montageChannelNames=montageChannelNames,
    parametersOfDenoiser=parametersOfDenoiser,
    verbose=False
)

# ==========================================
# Framework Component Initialization
# ==========================================
print("Initializing localized processing structures.")
windowing = OnEEGWaveLAD_Windowing(raw_data, RTWL=1000)
dwt = OnEEGWaveLAD_DWT(MW='sym4')

num_channels = len(raw_data.ch_names)
denoiser = OnEEGWaveLAD_Denoiser(n_channels=num_channels, Bs=20, IFS=512, Ta=0.55, Es=35)

target_channel_name = 'Fp1'
if target_channel_name in raw_data.ch_names:
    debug_ch_idx = raw_data.ch_names.index(target_channel_name)
else:
    debug_ch_idx = 0

target_debug_info = None
target_original_sig = None
target_denoised_sig = None
target_window_idx = -1

#Configuration for continuous snapshot temporal library (GIF generation)
moving_buffer_snapshots = []
moving_buffer_times = []

#Calculate the capture interval in windows (e.g., capturing one frame per second)
capture_interval_windows = max(1, int(1000 / windowing.actual_RTWL))

#Set the maximum simulation time to 60 seconds to prevent generating excessively large GIF files
max_simulation_time_sec = 60.0

# ==========================================
# Pseudo-Real-Time Data Stream Execution
# ==========================================
print(f"Executing continuous operational monitoring loop on channel: {raw_data.ch_names[debug_ch_idx]}")

#Initialization of arrays to record global anomaly scores
global_isp_scores = []
global_isp_times = []

for i, win in enumerate(windowing.get_window_stream()):
    coeffs = dwt.decompose_window(win['dwt_input'])

    denoised_signals, debug = denoiser.process_window(
        all_channels_coeffs=coeffs,
        target_len=win['original_window'].shape[1],
        return_debug=True,
        debug_channel=debug_ch_idx
    )

    current_time_sec = (i + 1) * windowing.actual_RTWL / 1000.0

    #Capture the moving buffer at the specified frequency for subsequent GIF and 3D visualizations
    if (i + 1) % capture_interval_windows == 0:
        moving_buffer_snapshots.append(debug['Step_D_Moving_Buffer'].copy())
        moving_buffer_times.append(current_time_sec)

    #Extract Instantaneous Spectral Profile (ISP) anomaly scores and map them to the global timeline
    scores = debug.get('Step_E_Scores')
    if scores is not None:
        global_isp_scores.extend(scores)

        #Calculate the corresponding continuous timestamp for each ISP within the current window
        t_start = i * windowing.actual_RTWL / 1000.0
        t_end = current_time_sec
        isp_t = np.linspace(t_start, t_end, len(scores), endpoint=False)
        global_isp_times.extend(isp_t)

    #Capture static diagnostic slices of detected structural anomalies
    if scores is not None and np.max(scores) > denoiser.Ta and target_debug_info is None:
        target_window_idx = i
        target_debug_info = debug
        target_original_sig = win['original_window'][debug_ch_idx]
        target_denoised_sig = denoised_signals[debug_ch_idx]
        print(f"Structural anomaly instance captured within frame index {i} (t={current_time_sec:.1f}s).")

    if current_time_sec >= max_simulation_time_sec:
        print(f"Required simulation time ({max_simulation_time_sec}s) reached. Terminating execution block.")
        break

Initiating dataset ingestion and pipeline parameters mapping.
Reading /content/drive/MyDrive/EEG_Project/N170 All Data and Scripts/1/1_N170_shifted.fdt
Reading 0 ... 699391  =      0.000 ...   682.999 secs...
Initializing localized processing structures.
[Phase A] window=1000.00 ms (1024 samples).
Executing continuous operational monitoring loop on channel: FP1
Structural anomaly instance captured within frame index 1 (t=2.0s).
Required simulation time (60.0s) reached. Terminating execution block.


In [8]:
# ==============================================================================
# CELL 7: Visualization 1: Statistical Visualization and Diagnostics (Static Plot)
# ==============================================================================
import os
import numpy as np
import matplotlib.pyplot as plt


if target_debug_info is not None:
    # Set up the figure with a 3x2 grid layout
    fig, axes = plt.subplots(3, 2, figsize=(16, 12))

    # Define a clean, academic global title containing specific parameters
    fig.suptitle(f"Six-Panel Diagnostic View of the onEEGwaveLAD Pipeline\n"
                 f"Subject 19, Channel {raw_data.ch_names[debug_ch_idx]}, "
                 f"Window {target_window_idx}, $B_s$ = {denoiser.Bs}",
                 fontsize=16, fontweight='bold', y=0.98)

    # Calculate time axes based on the actual window length (ms)
    time_axis = np.linspace(0, windowing.actual_RTWL, len(target_original_sig))
    isp_time_axis = np.linspace(0, windowing.actual_RTWL, len(target_debug_info['Step_E_Scores']))

    # Map axes variables for clearer code readability in a 2D grid
    ax_a, ax_b = axes[0, 0], axes[0, 1]
    ax_c, ax_d = axes[1, 0], axes[1, 1]
    ax_e, ax_f = axes[2, 0], axes[2, 1]

    # -------------------------------------------------------------------------
    # Panel (a): Comparison of Time-Domain Waveforms (Top-Left)
    # -------------------------------------------------------------------------
    ax_a.plot(time_axis, target_original_sig, label="Original Input (Contaminated)",
                 color='#e74c3c', alpha=0.7, linewidth=1.5)
    ax_a.plot(time_axis, target_denoised_sig, label="Denoised Output",
                 color='#2980b9', alpha=0.9, linewidth=1.5)
    ax_a.set_title("(a) Raw Input Window with Stimulus Markers", fontsize=13)
    ax_a.set_ylabel(r"Amplitude ($\mu V$)")
    ax_a.legend(loc='upper right', fontsize=10)
    ax_a.grid(True, linestyle='--', alpha=0.5)

    # -------------------------------------------------------------------------
    # Panel (b): Raw Discrete Wavelet Coefficients (Top-Right)
    # -------------------------------------------------------------------------
    sc_raw = target_debug_info['Step_C_Scaleogram_Raw'].T
    im_b = ax_b.imshow(sc_raw, aspect='auto', cmap='viridis', origin='lower',
                         extent=[0, windowing.actual_RTWL, 1, sc_raw.shape[0]], interpolation='none')
    ax_b.set_title("(b) Raw Scaleogram (Detail Coefficients)", fontsize=13)
    ax_b.set_ylabel("Decomposition Level")
    fig.colorbar(im_b, ax=ax_b, fraction=0.046, pad=0.04)

    # -------------------------------------------------------------------------
    # Panel (c): Normalized Scaleogram (ISPs) (Middle-Left)
    # -------------------------------------------------------------------------
    sc_before = target_debug_info['Step_C_Scaleogram_Before'].T
    im_c = ax_c.imshow(sc_before, aspect='auto', cmap='viridis', origin='lower',
                         extent=[0, windowing.actual_RTWL, 1, sc_before.shape[0]], interpolation='none')
    ax_c.set_title("(c) Normalized Scaleogram (ISPs)", fontsize=13)
    ax_c.set_ylabel("Decomposition Level")
    fig.colorbar(im_c, ax=ax_c, fraction=0.046, pad=0.04)

    # -------------------------------------------------------------------------
    # Panel (d): Extended Isolation Forest Anomaly Scores (Middle-Right)
    # -------------------------------------------------------------------------
    ax_d.plot(isp_time_axis, target_debug_info['Step_E_Scores'],
                 color='purple', linewidth=2, label='Isolation Anomaly Score')
    ax_d.axhline(y=denoiser.Ta, color='red', linestyle='--', linewidth=2,
                    label=f'Anomaly Threshold ($T_a$ = {denoiser.Ta})')
    ax_d.set_title("(d) Anomaly Scores from Extended Isolation Forest", fontsize=13)
    ax_d.set_ylabel("Score")
    ax_d.set_ylim(0, 1.05)
    ax_d.legend(loc='upper right', fontsize=10)
    ax_d.grid(True, linestyle='--', alpha=0.5)

    # -------------------------------------------------------------------------
    # Panel (e): Post-Attenuation Scaleogram (Bottom-Left)
    # -------------------------------------------------------------------------
    sc_after = target_debug_info['Step_C_Scaleogram_After'].T
    im_e = ax_e.imshow(sc_after, aspect='auto', cmap='viridis', origin='lower',
                         extent=[0, windowing.actual_RTWL, 1, sc_after.shape[0]], interpolation='none')
    ax_e.set_title("(e) Post-Mitigation Scaleogram", fontsize=13)
    ax_e.set_ylabel("Decomposition Level")
    ax_e.set_xlabel("Time (ms)")
    fig.colorbar(im_e, ax=ax_e, fraction=0.046, pad=0.04)

    # ==========================================
    # Panel (f): Artifacts Reduction Mitigator Array (Bottom-Right)
    # ==========================================
    mitigator = target_debug_info['Step_F_Mitigator'].T

    #Invert mitigator to show attenuation strength (more intuitive)
    attenuation_strength = 1.0 - mitigator

    im_f = ax_f.imshow(attenuation_strength, aspect='auto', cmap='Reds', origin='lower',
                        vmin=0, vmax=1,
                        extent=[0, windowing.actual_RTWL, 1, attenuation_strength.shape[0]],
                        interpolation='none')

    ax_f.set_title("(f) Attenuation Strength Map", fontsize=13)
    ax_f.set_ylabel("Decomposition Level")
    ax_f.set_xlabel("Time (ms)")

    cbar_f = fig.colorbar(im_f, ax=ax_f, fraction=0.046, pad=0.04)
    cbar_f.set_label("Attenuation (0=clean, 1=removed)", fontsize=10)

    #Add subtle grid to show data coverage
    ax_f.grid(True, color='black', linestyle=':', linewidth=0.3, alpha=0.15)

    # -------------------------------------------------------------------------
    # Finalize Layout and Export to Google Drive
    # -------------------------------------------------------------------------
    # Adjust spacing to prevent labels and titles from overlapping in the grid
    plt.tight_layout()
    plt.subplots_adjust(top=0.90, hspace=0.35, wspace=0.15)

    # Ensure Google Drive export directory exists
    try:
        drive_dir = '/content/drive/MyDrive/EEG_Project/Visualisations'
        os.makedirs(drive_dir, exist_ok=True)
        pdf_path = os.path.join(drive_dir, 'vis1_six_panel.pdf')
    except Exception:
        # Fallback to local directory if not in Colab or Drive isn't mounted
        pdf_path = 'vis1_six_panel.pdf'

    # Save the figure adhering to academic standards
    plt.savefig(pdf_path, dpi=300, bbox_inches='tight')

    # Close the figure to free memory constraints in loop executions
    plt.close()
    print(f"[SUCCESS] Visualization 1 (3x2 Grid) successfully exported to: {pdf_path}")

[SUCCESS] Visualization 1 (3x2 Grid) successfully exported to: /content/drive/MyDrive/EEG_Project/Visualisations/vis1_six_panel.pdf


In [9]:
# ==========================================
# CELL 8: Visualization 2: Moving Buffer Evolution - Static Frame and Animation
# ==========================================
import os
import shutil
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation

if len(moving_buffer_snapshots) > 0:
    print(f"\nProcessing {len(moving_buffer_snapshots)} captured buffer frames...")

    #Set up Google Drive export directory
    drive_dir = '/content/drive/MyDrive/EEG_Project/Visualisations'
    os.makedirs(drive_dir, exist_ok=True)

    #Global color bounds optimization (applies to both static and GIF)
    all_buffer_data = np.concatenate([b.flatten() for b in moving_buffer_snapshots])
    global_vmin = np.percentile(all_buffer_data, 1)
    global_vmax = np.percentile(all_buffer_data, 99)

    academic_cmap = 'turbo'

    #Part A: Generate and save a representative static PDF frame
    mid_frame_idx = len(moving_buffer_snapshots) // 2
    fig_static, ax_static = plt.subplots(figsize=(10, 7))
    buf_data_static = moving_buffer_snapshots[mid_frame_idx]
    t_static = moving_buffer_times[mid_frame_idx]

    im_static = ax_static.imshow(buf_data_static, aspect='auto', cmap=academic_cmap,
                                 interpolation='none', vmin=global_vmin, vmax=global_vmax)

    ax_static.set_ylabel("ISP Samples (Temporal History)", fontsize=13)
    ax_static.set_xlabel("Scaleogram Features", fontsize=13)

    ax_static.set_title(f"Sliding Buffer State at t = {t_static:.1f}s\n"
                        f"Subject 19, Channel {raw_data.ch_names[debug_ch_idx]}, "
                        f"$B_s$ = {denoiser.Bs}",
                        fontsize=14, fontweight='bold')

    cbar_static = fig_static.colorbar(im_static, ax=ax_static, fraction=0.02, pad=0.01)
    cbar_static.set_label("Normalized ISP Feature Amplitude", fontsize=12)

    plt.tight_layout()
    static_pdf_path = os.path.join(drive_dir, 'vis2_buffer_evolution.pdf')
    plt.savefig(static_pdf_path, dpi=300, bbox_inches='tight')
    plt.close(fig_static)
    print(f"[SUCCESS] Representative static frame saved to: {static_pdf_path}")

    #Part B: Generate and save the animated GIF
    fig_gif, ax_gif = plt.subplots(figsize=(8, 6))

    first_buffer = moving_buffer_snapshots[0]
    im_gif = ax_gif.imshow(first_buffer, aspect='auto', cmap=academic_cmap,
                           interpolation='none', vmin=global_vmin, vmax=global_vmax)

    ax_gif.set_ylabel("ISP Samples (Temporal History)", fontsize=12)
    ax_gif.set_xlabel("Scaleogram Features", fontsize=12)
    cbar = fig_gif.colorbar(im_gif, ax=ax_gif, fraction=0.02, pad=0.01)
    cbar.set_label("Normalized Feature Amplitude")

    title_text = ax_gif.set_title("", fontsize=14, fontweight='bold')

    def update_frame(frame_idx):
        buf_data = moving_buffer_snapshots[frame_idx]
        t = moving_buffer_times[frame_idx]
        im_gif.set_data(buf_data)
        title_text.set_text(f"Adaptive Moving Buffer Evolution\n"
                            f"Time: {t:.1f}s | Stored ISPs: {buf_data.shape[0]}")
        return [im_gif, title_text]

    #Create animation with blit=False to prevent background caching issues
    ani = animation.FuncAnimation(fig_gif, update_frame, frames=len(moving_buffer_snapshots), blit=False)
    local_gif_path = "/content/temp_vis2_buffer_evolution.gif"
    drive_gif_path = os.path.join(drive_dir, "vis2_buffer_evolution.gif")

    ani.save(local_gif_path, writer='pillow', fps=5)
    plt.close(fig_gif)
    shutil.copy(local_gif_path, drive_gif_path)

    #Clean up temporary local file
    if os.path.exists(local_gif_path):
        os.remove(local_gif_path)

    print(f"[SUCCESS] Animation successfully saved to: {drive_gif_path}")

else:
    print("[WARNING] Not enough frames captured in 'moving_buffer_snapshots' to generate a visualization.")


Processing 60 captured buffer frames...
[SUCCESS] Representative static frame saved to: /content/drive/MyDrive/EEG_Project/Visualisations/vis2_buffer_evolution.pdf
[SUCCESS] Animation successfully saved to: /content/drive/MyDrive/EEG_Project/Visualisations/vis2_buffer_evolution.gif


In [10]:
# ==============================================================================
# CELL 9: Visualization 3: 3D PCA Scatter Plot (Interactive Exploration & Static PDF)
# ==============================================================================
import os
import numpy as np
import plotly.graph_objects as go
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

if target_debug_info is not None:
    print("\nGenerating 3D PCA visualizations...")

    # Set up Google Drive export directory
    drive_dir = '/content/drive/MyDrive/EEG_Project/Visualisations'
    os.makedirs(drive_dir, exist_ok=True)

    # 1. Extracting Buffer Data and Re-predicting Scores
    buffer_data = target_debug_info['Step_D_Moving_Buffer']
    model = denoiser.models[debug_ch_idx]

    buffer_scores = model.predict(buffer_data)
    is_outlier = buffer_scores > denoiser.Ta

    # 2. PCA dimensionality reduction to 3 principal components
    pca = PCA(n_components=3)
    pca_result = pca.fit_transform(buffer_data)

    inliers_pca = pca_result[~is_outlier]
    outliers_pca = pca_result[is_outlier]

    # -------------------------------------------------------------------------
    # Part A: Generate Static Matplotlib Version for LaTeX (PDF Export)
    # -------------------------------------------------------------------------
    fig_static = plt.figure(figsize=(10, 8))
    ax_3d = fig_static.add_subplot(111, projection='3d')

    # Plot Inliers (Green markers)
    if len(inliers_pca) > 0:
        ax_3d.scatter(inliers_pca[:, 0], inliers_pca[:, 1], inliers_pca[:, 2],
                      c='#2ecc71', s=30, alpha=0.5, edgecolors='none',
                      label=f'Inliers (score $\leq T_a$)')

    # Plot Outliers (Red markers with distinct shape)
    if len(outliers_pca) > 0:
        ax_3d.scatter(outliers_pca[:, 0], outliers_pca[:, 1], outliers_pca[:, 2],
                      c='#e74c3c', s=50, alpha=0.9, edgecolors='black', linewidth=0.5,
                      marker='^', label=f'Outliers (score $> T_a$)')

    # Apply academic formatting to axes and titles
    # [FIX] Added labelpad=10 to pull the labels slightly closer to the axes
    ax_3d.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%} var.)', fontsize=12, labelpad=10)
    ax_3d.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%} var.)', fontsize=12, labelpad=10)
    ax_3d.set_zlabel(f'PC3 ({pca.explained_variance_ratio_[2]:.1%} var.)', fontsize=12, labelpad=10)

    # Adjust title position so it doesn't overlap with the top of the 3D box
    ax_3d.set_title(f'3D PCA Projection: Anomaly Separation at $T_a$ = {denoiser.Ta}\n'
                    f'Subject 19, Channel {raw_data.ch_names[debug_ch_idx]}, $B_s$ = {denoiser.Bs}',
                    fontsize=13, fontweight='bold', pad=30)

    ax_3d.legend(loc='upper left', fontsize=11)
    ax_3d.grid(True, alpha=0.3)

    # [FIX] Manually adjust the margins to leave enough space on the left/right for 3D labels
    # This prevents the Z-axis (PC3) or Y-axis text from being clipped in the PDF
    fig_static.subplots_adjust(left=0.1, right=0.9, top=0.9, bottom=0.1)

    # Export to PDF (removed bbox_inches='tight' as it often fights with 3D axes padding)
    static_pdf_path = os.path.join(drive_dir, 'vis3_pca_separation.pdf')
    plt.savefig(static_pdf_path, dpi=300)
    plt.close(fig_static) # Free memory
    print(f"[SUCCESS] Static LaTeX-ready PDF saved to: {static_pdf_path}")

    # -------------------------------------------------------------------------
    # Part B: Build Interactive 3D Chart using Plotly (for Notebook Exploration)
    # -------------------------------------------------------------------------
    print("\nRendering Interactive 3D PCA Cube... (Use your mouse to rotate/zoom)")
    fig3d = go.Figure()

    # Draw normal Inliers (green dots)
    if len(inliers_pca) > 0:
        fig3d.add_trace(go.Scatter3d(
            x=inliers_pca[:, 0], y=inliers_pca[:, 1], z=inliers_pca[:, 2],
            mode='markers',
            marker=dict(
                size=4,
                color='#2ecc71',
                opacity=0.5
            ),
            name=f'Inliers (Scores <= {denoiser.Ta})'
        ))

    # Draw the outliers that indicate anomalies (red dots)
    if len(outliers_pca) > 0:
        fig3d.add_trace(go.Scatter3d(
            x=outliers_pca[:, 0], y=outliers_pca[:, 1], z=outliers_pca[:, 2],
            mode='markers',
            marker=dict(
                size=6,
                color='#e74c3c',
                opacity=0.9,
                line=dict(width=1, color='black')
            ),
            name=f'Outliers (Scores > {denoiser.Ta})'
        ))

    # Configure interactive layout and axis labels
    fig3d.update_layout(
        title=dict(
            text="Interactive 3D PCA Projection of Adaptive Moving Buffer (ISPs)",
            font=dict(size=18, color='black'),
            x=0.5,
            y=0.95
        ),
        scene=dict(
            xaxis_title=f"PC 1 ({pca.explained_variance_ratio_[0]:.1%} var)",
            yaxis_title=f"PC 2 ({pca.explained_variance_ratio_[1]:.1%} var)",
            zaxis_title=f"PC 3 ({pca.explained_variance_ratio_[2]:.1%} var)",
            xaxis=dict(backgroundcolor="white", gridcolor="lightgrey"),
            yaxis=dict(backgroundcolor="white", gridcolor="lightgrey"),
            zaxis=dict(backgroundcolor="white", gridcolor="lightgrey"),
        ),
        legend=dict(
            yanchor="top",
            y=0.9,
            xanchor="left",
            x=0.05
        ),
        margin=dict(l=0, r=0, b=0, t=0),
        width=900,
        height=700
    )

    # Display interactive charts directly in Notebook
    fig3d.show()

else:
    print("[WARNING] Cannot generate PCA plot: No target buffer data was captured.")

<>:42: SyntaxWarning: invalid escape sequence '\l'
<>:42: SyntaxWarning: invalid escape sequence '\l'
/tmp/ipykernel_22692/3531254501.py:42: SyntaxWarning: invalid escape sequence '\l'
  label=f'Inliers (score $\leq T_a$)')



Generating 3D PCA visualizations...
[SUCCESS] Static LaTeX-ready PDF saved to: /content/drive/MyDrive/EEG_Project/Visualisations/vis3_pca_separation.pdf

Rendering Interactive 3D PCA Cube... (Use your mouse to rotate/zoom)


In [11]:
# ==============================================================================
# CELL 10: Visualization 4: Global Anomaly Score Trend (Over the entire signal)
# ==============================================================================
import os
import matplotlib.pyplot as plt

if len(global_isp_scores) > 0:
    print("\nGenerating Global Anomaly Score Trend...")

    # Set up Google Drive export directory
    drive_dir = '/content/drive/MyDrive/EEG_Project/Visualisations'
    os.makedirs(drive_dir, exist_ok=True)

    # Create a wide-format figure for extended timeline visualization
    fig_global, ax_global = plt.subplots(figsize=(16, 4))

    # Plot the global anomaly score trajectory
    ax_global.plot(global_isp_times, global_isp_scores, color='purple',
                   linewidth=1, alpha=0.8, label='ISP Anomaly Score')

    # Plot the anomaly decision threshold Ta
    ax_global.axhline(y=denoiser.Ta, color='red', linestyle='--', linewidth=2,
                      label=f'Anomaly Threshold ($T_a$ = {denoiser.Ta})')

    # Highlight: Apply shaded regions to distinguish between clean and contaminated zones
    ax_global.fill_between(global_isp_times, 0, denoiser.Ta, color='#2ecc71',
                           alpha=0.1, label='Clean EEG Zone (Inliers)')
    ax_global.fill_between(global_isp_times, denoiser.Ta, 1.05, color='#e74c3c',
                           alpha=0.1, label='Contaminated Zone (Artefacts)')

    # Configure the plot aesthetics with academic titling
    ax_global.set_title(f"Global Anomaly Score Evolution Over Time\n"
                        f"Subject 19, Channel {raw_data.ch_names[debug_ch_idx]}, $B_s$ = {denoiser.Bs}",
                        fontsize=15, fontweight='bold', pad=15)

    ax_global.set_xlabel("Time (seconds)", fontsize=13)
    ax_global.set_ylabel("Isolation Forest Score", fontsize=13)
    ax_global.set_ylim(0, 1.05)
    ax_global.set_xlim(0, max(global_isp_times))

    # Add legend and grid lines
    ax_global.legend(loc='upper right', fontsize=11)
    ax_global.grid(True, linestyle=':', alpha=0.7)

    plt.tight_layout()

    # Export the figure to Google Drive as a high-resolution PDF
    pdf_path = os.path.join(drive_dir, 'vis4_anomaly_trajectory.pdf')
    plt.savefig(pdf_path, dpi=300, bbox_inches='tight')

    # Close the figure to free up memory
    plt.close(fig_global)
    print(f"[SUCCESS] Global Anomaly Score Trend successfully exported to: {pdf_path}")

else:
    print("[WARNING] No global scores were recorded.")


Generating Global Anomaly Score Trend...
[SUCCESS] Global Anomaly Score Trend successfully exported to: /content/drive/MyDrive/EEG_Project/Visualisations/vis4_anomaly_trajectory.pdf


In [12]:
# ==============================================================================
# CELL 11: Visualization 5: Clean vs. Contaminated Buffer Distributions
# ==============================================================================
import os
import numpy as np
import matplotlib.pyplot as plt

if len(moving_buffer_snapshots) > 0 and target_debug_info is not None:
    print("\nGenerating Distribution Comparison (Clean vs. Contaminated)...")

    # Set up Google Drive export directory
    drive_dir = '/content/drive/MyDrive/EEG_Project/Visualisations'
    os.makedirs(drive_dir, exist_ok=True)

    # Retrieve the corresponding Isolation Forest model
    model = denoiser.models[debug_ch_idx]

    # 1. Iterate over all captured buffer snapshots and calculate the anomaly scores
    snapshot_scores = []
    for buf in moving_buffer_snapshots:
        scores = model.predict(buf)
        snapshot_scores.append(scores)

    # 2. Automatically identify the indices of the buffers representing the two extreme states
    # Cleanest state: The moment when the maximum anomaly score within the buffer is at its minimum
    cleanest_idx = np.argmin([np.max(scores) for scores in snapshot_scores])
    # Most contaminated state: The moment when the maximum anomaly score within the buffer reaches its peak
    contaminated_idx = np.argmax([np.max(scores) for scores in snapshot_scores])

    clean_scores = snapshot_scores[cleanest_idx]
    contam_scores = snapshot_scores[contaminated_idx]
    t_clean = moving_buffer_times[cleanest_idx]
    t_contam = moving_buffer_times[contaminated_idx]

    # 3. Plot overlapping histograms to compare the distributions
    fig_dist, ax_dist = plt.subplots(figsize=(12, 6))

    # Plot the distribution of the clean state (green)
    ax_dist.hist(clean_scores, bins=40, color='#2ecc71', alpha=0.7,
                 label=f'Relatively Clean Buffer (t = {t_clean:.1f}s)',
                 density=True, edgecolor='white')

    # Plot the distribution of the contaminated state (red)
    ax_dist.hist(contam_scores, bins=40, color='#e74c3c', alpha=0.7,
                 label=f'Highly Contaminated Buffer (t = {t_contam:.1f}s)',
                 density=True, edgecolor='white')

    # Plot the anomaly decision threshold Ta
    ax_dist.axvline(x=denoiser.Ta, color='red', linestyle='--', linewidth=2,
                    label=f'Anomaly Threshold ($T_a$ = {denoiser.Ta})')

    # Configure the academic title and axis labels
    ax_dist.set_title(f"Distribution of Anomaly Scores: Clean vs. Contaminated Buffer States\n"
                      f"Subject 19, Channel {raw_data.ch_names[debug_ch_idx]}, $B_s$ = {denoiser.Bs}",
                      fontsize=14, fontweight='bold', pad=15)

    ax_dist.set_xlabel("Isolation Forest Anomaly Score", fontsize=13)
    ax_dist.set_ylabel("Density (Proportion of ISPs)", fontsize=13)

    # Note: Text annotations (RQ1/RQ2) were removed to maintain objective academic styling

    ax_dist.legend(loc='upper right', fontsize=11)
    ax_dist.grid(True, linestyle=':', alpha=0.6)

    plt.tight_layout()

    # Export the figure to Google Drive as a high-resolution PDF
    pdf_path = os.path.join(drive_dir, 'vis5_score_distribution.pdf')
    plt.savefig(pdf_path, dpi=300, bbox_inches='tight')

    # Close the figure to free up memory
    plt.close(fig_dist)
    print(f"[SUCCESS] Distribution Comparison successfully exported to: {pdf_path}")

else:
    print("[WARNING] Not enough data to generate the distribution comparison. Ensure the simulation ran long enough.")


Generating Distribution Comparison (Clean vs. Contaminated)...
[SUCCESS] Distribution Comparison successfully exported to: /content/drive/MyDrive/EEG_Project/Visualisations/vis5_score_distribution.pdf


In [13]:
# ==============================================================================
# CELL 12: Baseline Method Dependencies Installation
# ==============================================================================

!pip install scikit-posthocs
!pip install statsmodels
!pip install asrpy hurst
!pip install scikit-posthocs
!pip install statsmodels
!pip install asrpy hurst

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.4/45.4 kB 1.7 MB/s eta 0:00:00


In [14]:
# ==============================================================================
# CELL 13: Baseline Method - FASTER-Based ICA Component Rejection
# ==============================================================================
import timeit
import numpy as np
import scipy as sp
import matplotlib.pyplot as plt
import mne
from hurst import compute_Hc
from asrpy.asr_utils import (geometric_median, fit_eeg_distribution,
                             yulewalk, yulewalk_filter, ma_filter, block_covariance)

#1st baseline -> FASTER-INSPIRED
#===============================

def computeIndependentComponentsProperties(ica_sources, numOfComponents, eegData, sampleFreq, minFreq, maxFreq, EOGData, verbose=False):

  #https://www.codespeedy.com/hurst-exponent-in-python/

  #SP-K: spatial kurtosis of the time serie of an independent component
  #SL-F: mean of the slope (gradient) in the filter band defined by the low pass filter edges (eg. 1-128 Hz)
  #HURST: hurst exponent of the time serie of an independent component
  #MG: median of the slope (gradient) of the time serie of an independent component
  #EOG-CORR: correlation of the time serie of an independent component with an EOG time serie

  EOGDataTimeSerie = EOGData.get_data()[0]
  if (verbose):
    print("Computing properties of {} independent components, with data sampled at {} hz, with high and low pass frequency thresholds  {}-{}".format(numOfComponents, sampleFreq, minFreq, maxFreq))
    print ("The shape of EOG data is: {}".format(EOGData.get_data().shape))
    print ("The shape of EOG data is: {}".format(EOGData.get_data()[0].shape))

  compProperties = [None]*numOfComponents
  EOGCORRVals = [0]*numOfComponents
  SPKVals  = [0]*numOfComponents
  SLFVals = [0]*numOfComponents
  HURSTVals = [0]*numOfComponents
  MGVals = [0]*numOfComponents

  for comp in range(numOfComponents):
    properties = {
      "SP-K":0, "SL-F":0, "HURST":0, "MG":0,
      "Z-SP-K":0, "Z-SL-F":0, "Z-HURST":0, "Z-MG":0, "Z-EOG-CORR":0
    }

    c = ica_sources.get_data(picks=[comp]) # Shape: (1, n_times)
    component_time_series = c[0]

    if (verbose):
      plt.plot(eegData.times, component_time_series)
      plt.xlabel("Time (s) [0..{}]".format(len(eegData.times)))
      plt.ylabel("Activation")
      plt.title("ICA Component {}".format(comp))
      plt.show()
      plt.close()
      plt.close('all')

    #Z-scores holders for each property of an independent component

    #compute spatial kurtosis for the current component
    SPKVals[comp] = properties ["SP-K"] = sp.stats.kurtosis(component_time_series)

    #compute power spectrum for the current component
    fourier_transform = np.fft.rfft(component_time_series)
    abs_fourier_transform = np.abs(fourier_transform)
    power_spectrum = np.square(abs_fourier_transform)
    frequency = np.linspace(minFreq, maxFreq, len(power_spectrum))
    SLFVals[comp] = properties ["SL-F"] = np.mean(np.gradient(power_spectrum))
    if (verbose):
      print ("LENS - FT: {}, PS: {}, freq: {}".format(len(fourier_transform), len(power_spectrum), len(frequency)))
      plt.plot(power_spectrum)
      plt.title("Power spectrum (Component: {})".format(comp))
      plt.show()
      plt.close()
      plt.close('all')

    #compute HURST exponent for the current component
    #compute_Hc returns a tuple of 3 values (h=Hurst exponent, c=constant, val=values of the Hurst exponent)
    h, c, val = compute_Hc(component_time_series)
    HURSTVals[comp] = properties ["HURST"] = h
    if (verbose):
      # Plot the graph
      axes = plt.subplots()[1]
      axes.plot(val[0], c*val[0]**h, color="blue")
      axes.scatter(val[0], val[1], color="red")
      axes.set_xscale('log')
      axes.set_yscale('log')
      axes.set_xlabel('Time interval')
      axes.set_ylabel('R/S ratio')
      axes.grid(True)
      plt.title("HURST exponent (Component: {})".format(comp))
      plt.show()
      plt.close()
      plt.close('all')

    #compute median slope for the current component
    MGVals[comp] = properties ["MG"] = np.median(np.gradient(component_time_series))

    #compute correlation of current component with EOG signals
    #EOGCORRVals[comp] = properties ["HEOGL-CORR"] = np.corrcoef(component_time_series, EOGDataTimeSerie)[0]
    EOGCORRVals[comp] = properties ["HEOGL-CORR"] = sp.stats.pearsonr(component_time_series, EOGDataTimeSerie)[0]

    #assign computed properties for current component to list of components
    compProperties[comp] = properties

  #compute z-scores for each property
  zSPKVals = sp.stats.zscore(SPKVals)
  zSLFVals = sp.stats.zscore(SLFVals)
  zHURSTVals = sp.stats.zscore(HURSTVals)
  zMGVals = sp.stats.zscore(MGVals)
  zEOGCORRVals = sp.stats.zscore(EOGCORRVals)

  #print z-scores of each property
  if (verbose):
    plt.plot(zSPKVals, color="blue", label="spatial kurtosis")
    plt.plot(zSLFVals, color="red", label="mean of the slope")
    plt.plot(zHURSTVals, color="orange", label="hurst exponent")
    plt.plot(zMGVals, color="green", label="median of the slope")
    plt.plot(zEOGCORRVals, color="violet", label="correlation with EOG")
    plt.xlabel("Component")
    plt.ylabel("z-score")
    plt.title("Z-scores of properties")
    plt.legend()
    plt.show()
    plt.close()

  for comp in range(numOfComponents):
     compProperties[comp]["Z-SP-K"] = zSPKVals[comp]
     compProperties[comp]["Z-SL-F"] = zSLFVals[comp]
     compProperties[comp]["Z-HURST"] = zHURSTVals[comp]
     compProperties[comp]["Z-MG"] = zMGVals[comp]
     compProperties[comp]["Z-EOG-CORR"] = zEOGCORRVals[comp]


  return compProperties



def runICARemoveComponentsRunInverseICA(eegData, baselineLenMs, ERPLenMs, sampleFreq, numComponents,
    highPassTh, lowPassTh, EOGData, removeComponentsWithFASTERMethod = False, plotComponents=False, verbose=False):

  #maxExplainedVarianceForICA = 0.99
  numComponents = numComponents-1
  ICAStart = timeit.default_timer()
  ica = mne.preprocessing.ICA(max_iter="auto", random_state=97)
  ica.fit(eegData)
  ICAStop = timeit.default_timer()
  executionTime = ICAStop - ICAStart
  explained_var_ratio = ica.get_explained_variance_ratio(eegData)
  ica_sources = ica.get_sources(eegData)
  numOfICAComps = ica.n_components_

  #compute properties for all the components (with the FASTER methodology)
  compProperties = computeIndependentComponentsProperties(ica_sources,
      numOfICAComps, eegData, sampleFreq, highPassTh, lowPassTh, EOGData, verbose=False)

  #loop through components to spot those ones potentially containing artefacts (using the FASTER methodology)
  artifactual_components = []

  threshold = 3 #3 std
  if (removeComponentsWithFASTERMethod):
    for comp in range(numOfICAComps):
      c = compProperties[comp]
      if (c["Z-SP-K"]>threshold or c["Z-SL-F"]>threshold or c["Z-HURST"]>threshold or c["Z-MG"]>threshold or c["Z-EOG-CORR"]>threshold or
          c["Z-SP-K"]<-threshold or c["Z-SL-F"]<-threshold or c["Z-HURST"]<-threshold or c["Z-MG"]<-threshold or c["Z-EOG-CORR"]<-threshold):
        artifactual_components.append(comp)

  #run inverse ICA
  ica.exclude = artifactual_components  # Mark components for exclusion
  clean_eeg = ica.apply(eegData.copy())  # Reconstruct signal with excluded components (it is an MNE object)

  #plot independent ICA components as sources (timeseries) and topographic maps
  if (plotComponents):
    print ("Plot components")
    ica.plot_sources(eegData, start=0, title="Original EEG", show_scrollbars=False)
    ica.plot_components()

  if (verbose):
    print ("Number of ICA components computed is: {}".format(numOfICAComps))
    print ("ICA execution time is: {}".format(executionTime))
    component_time_series = ica_sources.get_data(picks=[0])  # Shape: (1, n_times)
    print ("Shape of a single component is: {}".format(component_time_series.shape))
    print ("Component time series shape: {}".format(component_time_series.shape))
    print ("Artefactual Components are {} with indexes: {}".format(len(artifactual_components), artifactual_components))
    for channel_type, ratio in explained_var_ratio.items():
      print(f"Fraction of {channel_type} variance explained by all components: {ratio}")

  return clean_eeg, len(artifactual_components)


#2nd baseline -> FASTER-INSPIRED
#===============================
  def denoiseWithArtifaceSuspaceReconstructionwithASROffline(rawDATA, sampleFreq, n_channels, cutOffASR,
                                         window_size_sec, channelNames, precleaned=True, verbose=False):

    #rawDATA = dataMNEObject.get_data()
    lenOfRawData = len (rawDATA[0])
    maxWindows = int((len(rawDATA[0])/sampleFreq) / window_size_sec)  #retrieve the max EEG win of window_size_sec to clean individually
    fullCleanEEGData = np.zeros((n_channels, lenOfRawData), dtype = 'float')  #container of clearn data
    lengthToplot = int(lenOfRawData/6)

    # (optional) make sure your asr is only fitted to clean parts of the data (this is taken from the whole data)
    calibrationData = rawDATA #the whole data is used, since it is offline, and this is possible

    maxAmp = np.max(calibrationData.max(axis=1))
    minAmp  = np.min(calibrationData.min(axis=1))

    if (verbose):
      print ("Calibration data shape before cleaning: {}".format(calibrationData.shape))
      plt.plot(calibrationData[:,0:lengthToplot])
      plt.ylim(bottom=minAmp)
      plt.ylim(top=maxAmp)
      plt.show()


    if (precleaned):
      min_clean_fraction = 0.25
      max_dropout_fraction = 0.1
      calibrationDataCleaned, _ = clean_windows(calibrationData, sampleFreq, max_bad_chans=0.05, win_len=window_size_sec, min_clean_fraction=min_clean_fraction, max_dropout_fraction=max_dropout_fraction)

    if (verbose):
      print ("Calibration data shape after cleaning: {}".format(calibrationData.shape))
      plt.plot(calibrationDataCleaned[:,0:lengthToplot])
      plt.ylim(bottom=minAmp)
      plt.ylim(top=maxAmp)
      plt.show()

    # fit the ASR
    M, T = asr_calibrate(calibrationDataCleaned, sampleFreq, cutoff=cutOffASR, win_len=window_size_sec)

    if (verbose):
      print ("Full Raw data before ASR: {}".format(rawDATA.shape))
      plt.plot(rawDATA[:,0:lengthToplot])
      plt.ylim(bottom=minAmp)
      plt.ylim(top=maxAmp)
      plt.show()

    # clean data
    clean_array = asr_process(rawDATA, sampleFreq, M, T, windowlen=window_size_sec)

    if (verbose):
      print ("Full Raw data after ASR: {}".format(clean_array.shape))
      plt.plot(clean_array[:,0:lengthToplot])
      plt.ylim(bottom=minAmp)
      plt.ylim(top=maxAmp)
      plt.show()

    return clean_array

  def denoiseWithArtifaceSuspaceReconstructionwithASRRealTimeLogic(rawDATA, sampleFreq, calibLengthInPoints, n_channels,
                            cutOffASR, window_size_sec, channelNames, precleaned=False, verbose=False):

    #rawDATA = dataMNEObject.get_data()
    maxWindows = int((len(rawDATA[0])/sampleFreq) / window_size_sec)  #retrieve the max EEG win of window_size_sec to clean individually
    fullCleanEEGData = np.zeros((n_channels, len (rawDATA[0])), dtype = 'float')  #container of clean data

    #create a numpy array of EEG data from the MNE raw object
    calibrationData = rawDATA[:,0:calibLengthInPoints]

    # (optional) make sure  asr is only fitted to clean parts of the calibration data
    if (precleaned):
      min_clean_fraction = 0.35
      max_dropout_fraction = 0.05
      pre_cleaned, _ = clean_windows(calibrationData, sampleFreq, max_bad_chans=0.1, win_len=window_size_sec, min_clean_fraction=min_clean_fraction, max_dropout_fraction=max_dropout_fraction)
      calibrationData = pre_cleaned

    # fit the asr only with the calibration data
    #bs = len(calibrationData[0]) #the length in points of the calibration interval
    M, T = asr_calibrate(calibrationData, sampleFreq, cutoff=cutOffASR, win_len=window_size_sec, min_clean_fraction=min_clean_fraction, max_dropout_fraction=max_dropout_fraction)

    # ---- Real-time processing loop ----
    for win in range(maxWindows):  #loop through all the EEG windows
      windowData = np.zeros((n_channels, int(sampleFreq)), dtype = 'float') #container of single window
      start = int(win*sampleFreq)
      stop = int((win+1)*sampleFreq)

      if (win == (maxWindows-1)):
        windowData = rawDATA[:, start:] #last window
      else:
        windowData = rawDATA[:, start:stop]

      #clean the current window
      clean_win = asr_process(windowData, sampleFreq, M, T, windowlen=window_size_sec)

      fullCleanEEGData[:, start:stop] = clean_win

    return fullCleanEEGData


# Authors:  Dirk Gütlin <dirk.guetlin@gmail.com>
#           Nicolas Barascud
#
# License: BSD (3-clause)
#https://github.com/DiGyt/asrpy/blob/main/example.ipynb

"""
In asrpy.asr you can find the original ASR functions (similar to MATLAB)
as well as a high-level ASR object ready to use with MNE-Python raw data.
"""
import logging
import warnings

import numpy as np
from scipy import linalg
from numpy.linalg import pinv

#from asr_utils import (geometric_median, fit_eeg_distribution, yulewalk, yulewalk_filter, ma_filter, block_covariance)


class ASR():
    """Artifact Subspace Reconstruction.

    Artifact subspace reconstruction (ASR) is an automated, online,
    component-based artifact removal method for removing transient or
    large-amplitude artifacts in multi-channel EEG recordings [1]_.

    Parameters
    ----------
    sfreq : float
        Sampling rate of the data, in Hz.
    cutoff: float
        Standard deviation cutoff for rejection. X portions whose variance
        is larger than this threshold relative to the calibration data are
        considered missing data and will be removed. The most aggressive value
        that can be used without losing too much EEG is 2.5. Recommended to
        use with more conservative values ranging from 20 - 30.
        Defaults to 20.
    blocksize : int
        Block size for calculating the robust data covariance and thresholds,
        in samples; allows to reduce the memory and time requirements of the
        robust estimators by this factor (down to Channels x Channels x Samples
        x 16 / Blocksize bytes) (default=100).
    win_len : float
        Window length (s) that is used to check the data for artifact content.
        This is ideally as long as the expected time scale of the artifacts but
        not shorter than half a cycle of the high-pass filter that was used
        (default=0.5).
    win_overlap : float
        Window overlap fraction. The fraction of two successive windows that
        overlaps. Higher overlap ensures that fewer artifact portions are going
        to be missed, but is slower (default=0.66).
    max_dropout_fraction : float
        Maximum fraction of windows that can be subject to signal dropouts
        (e.g., sensor unplugged), used for threshold estimation (default=0.1).
    min_clean_fraction : float
        Minimum fraction of windows that need to be clean, used for threshold
        estimation (default=0.25).
    ab : 2-tuple | None
        Coefficients (A, B) of an IIR filter that is used to shape the
        spectrum of the signal when calculating artifact statistics. The
        output signal does not go through this filter. This is an optional way
        to tune the sensitivity of the algorithm to each frequency component
        of the signal. The default filter is less sensitive at alpha and beta
        frequencies and more sensitive at delta (blinks) and gamma (muscle)
        frequencies. Defaults to None.
    max_bad_chans : float
        The maximum number or fraction of bad channels that a retained window
        may still contain (more than this and it is removed). Reasonable range
        is 0.05 (very clean output) to 0.3 (very lax cleaning of only coarse
        artifacts) (default=0.2).
    method : {'riemann', 'euclid'}
        Method to use. If riemann, use the riemannian-modified version of
        ASR [2]_. Currently, only euclidean ASR is supported. Defaults to
        "euclid".

    Attributes
    ----------
    sfreq: array, shape=(n_channels, filter_order)
        Filter initial conditions.
    cutoff: float
        Standard deviation cutoff for rejection.
    blocksize : int
        Block size for calculating the robust data covariance and thresholds.
    win_len : float
        Window length (s) that is used to check the data for artifact content.
    win_overlap : float
        Window overlap fraction.
    max_dropout_fraction : float
        Maximum fraction of windows that can be subject to signal dropouts.
    min_clean_fraction : float
        Minimum fraction of windows.
    max_bad_chans : float
        The maximum fraction of bad channels.
    method : {'riemann', 'euclid'}
        Method to use.
    A, B: arrays
        Coefficients of an IIR filter that is used to shape the spectrum of the
        signal when calculating artifact statistics. The output signal does not
        go through this filter. This is an optional way to tune the sensitivity
        of the algorithm to each frequency component of the signal. The default
        filter is less sensitive at alpha and beta frequencies and more
        sensitive at delta (blinks) and gamma (muscle) frequencies.
    M : array, shape=(channels, channels)
        The mixing matrix to fit ASR data.
    T : array, shape=(channels, channels)
        The mixing matrix to fit ASR data.

    References
    ----------
    .. [1] Kothe, C. A. E., & Jung, T. P. (2016). U.S. Patent Application No.
       14/895,440. https://patents.google.com/patent/US20160113587A1/en
    .. [2] Blum, S., Jacobsen, N. S. J., Bleichner, M. G., & Debener, S.
       (2019). A Riemannian Modification of Artifact Subspace Reconstruction
       for EEG Artifact Handling. Frontiers in Human Neuroscience, 13.
       https://doi.org/10.3389/fnhum.2019.00141

    """

    def __init__(self, sfreq, cutoff=20, blocksize=100, win_len=0.5,
                 win_overlap=0.66, max_dropout_fraction=0.1,
                 min_clean_fraction=0.25, ab=None, max_bad_chans=0.1,
                 method="euclid"):

        # set attributes
        self.sfreq = sfreq
        self.cutoff = cutoff
        self.blocksize = blocksize
        self.win_len = win_len
        self.win_overlap = win_overlap
        self.max_dropout_fraction = max_dropout_fraction
        self.min_clean_fraction = min_clean_fraction
        self.max_bad_chans = max_bad_chans
        self.method = "euclid"  # NOTE: riemann is not yet available
        self._fitted = False

        # set default yule-walker filter
        if ab is None:
            yw_f = np.array([0, 2, 3, 13, 16, 40,
                             np.minimum(80.0, (self.sfreq / 2.0) - 1.0),
                             self.sfreq / 2.0]) * 2.0 / self.sfreq
            yw_m = np.array([3, 0.75, 0.33, 0.33, 1, 1, 3, 3])
            self.B, self.A = yulewalk(8, yw_f, yw_m)
        else:
            self.A, self.B = ab

        self._reset()

    def _reset(self):
        """Reset state variables."""
        self.M = None
        self.T = None

        # TODO: The following parameters are effectively not used. Still,
        #  they can be set manually via asr.transform(return_states=True)
        self.R = None
        self.carry = None
        self.Zi = None
        self.cov = None
        self._fitted = False

    def fit(self, raw, picks="eeg", start=0, stop=None,
            return_clean_window=False):
        """Calibration for the Artifact Subspace Reconstruction method.

        The input to this data is a multi-channel time series of calibration
        data. In typical uses the calibration data is clean resting EEG data
        of data if the fraction of artifact content is below the breakdown
        point of the robust statistics used for estimation (50% theoretical,
        ~30% practical). If the data has a proportion of more than 30-50%
        artifacts then bad time windows should be removed beforehand. This
        data is used to estimate the thresholds that are used by the ASR
        processing function to identify and remove artifact components.

        The calibration data must have been recorded for the same cap design
        from which data for cleanup will be recorded, and ideally should be
        from the same session and same subject, but it is possible to reuse
        the calibration data from a previous session and montage to the
        extent that the cap is placed in the same location (where loss in
        accuracy is more or less proportional to the mismatch in cap
        placement).

        Parameters
        ----------
        raw : instance of mne.io.Raw
            Instance of mne.io.Raw to be used for fitting the ASR.
            The calibration data should have been high-pass filtered (for
            example at 0.5Hz or 1Hz using a Butterworth IIR filter), and be
            reasonably clean not less than 30 seconds (this method is
            typically used with 1 minute or more).
        picks : str | list | slice | None
            Channels used to fit the ASR. All channels should be of the same
            type (e.g. "eeg", "grads"). Slices and lists of integers will
            be interpreted as channel indices. In lists, channel
            name strings (e.g., ['MEG0111', 'MEG2623'] will pick the given
            channels. Note that channels in info['bads'] will be included if
            their names or indices are explicitly provided. Defaults to "eeg".
        start : int
            The first sample to use for fitting the data. Defaults to 0.
        stop : int | None
            The last sample to use for fitting the data. If `None`, all
            samples after `start` will be used for fitting. Defaults to None.
        return_clean_window : Bool
            If True, the method will return the variables `clean` (the cropped
             dataset which was used to fit the ASR) and `sample_mask` (a
             logical mask of which samples were included/excluded from fitting
             the ASR). Defaults to False.

        Returns
        -------
        clean : array, shape=(n_channels, n_samples)
            The cropped version of the dataset which was used to calibrate
            the ASR. This array is a result of the `clean_windows` function
            and no ASR was applied to it.
        sample_mask : boolean array, shape=(1, n_samples)
            Logical mask of the samples which were used to train the ASR.

        """

        # extract the data
        X = raw.get_data(picks=picks, start=start, stop=stop)

        # Find artifact-free windows first
        clean, sample_mask = clean_windows(
            X,
            sfreq=self.sfreq,
            win_len=self.win_len,
            win_overlap=self.win_overlap,
            max_bad_chans=self.max_bad_chans,
            min_clean_fraction=self.min_clean_fraction,
            max_dropout_fraction=self.max_dropout_fraction)

        # Perform calibration
        self.M, self.T = asr_calibrate(
            clean,
            sfreq=self.sfreq,
            cutoff=self.cutoff,
            blocksize=self.blocksize,
            win_len=self.win_len,
            win_overlap=self.win_overlap,
            max_dropout_fraction=self.max_dropout_fraction,
            min_clean_fraction=self.min_clean_fraction,
            ab=(self.A, self.B),
            method=self.method)

        self._fitted = True

        # return data if required
        if return_clean_window:
            return clean, sample_mask

    def transform(self, raw, picks="eeg", lookahead=0, stepsize=32,
                  maxdims=0.66, return_states=False, mem_splits=3):
        """Apply Artifact Subspace Reconstruction.

        Parameters
        ----------
        raw : instance of mne.io.Raw
            Instance of mne.io.Raw to be transformed by the ASR.
        picks : str | list | slice | None
            Channels to be transformed by the ASR. Should be the same set of
            channels as used by `ASR.fit()`. All channels should be of the
            same type (e.g. "eeg", "grads"). Slices and lists of integers will
            be interpreted as channel indices. In lists, channel
            name strings (e.g., ['MEG0111', 'MEG2623'] will pick the given
            channels. Note that channels in info['bads'] will be included if
            their names or indices are explicitly provided. Defaults to "eeg".
        lookahead : float
            Amount of look-ahead that the algorithm should use (in seconds).
            This value should be between 0 (no lookahead) and WindowLength/2
            (optimal lookahead). The recommended value is WindowLength/2.
            Default: 0.25

            Note: Other than in `asr_process`, the signal will be readjusted
            to eliminate any temporal jitter and automatically readjust it to
            the correct time points. Zero-padding will be applied to the last
            `lookahead` portion of the data, possibly resulting in inaccuracies
            for the final `lookahead` seconds of the recording.
        stepsize : int
            The steps in which the algorithm will be updated. The larger this
            is, the faster the algorithm will be. The value must not be larger
            than WindowLength * SamplingRate. The minimum value is 1 (update
            for every sample) while a good value would be sfreq//3. Note that
            an update is always performed also on the first and last sample of
            the data chunk. Default: 32
        max_dims : float, int
            Maximum dimensionality of artifacts to remove. This parameter
            denotes the maximum number of dimensions which can be removed from
            each segment. If larger than 1, `int(max_dims)` will denote the
            maximum number of dimensions removed from the data. If smaller
            than 1, `max_dims` describes a fraction of total dimensions.
            Defaults to 0.66.
        return_states : bool
            If True, returns a dict including the updated states {"M":M,
            "T":T, "R":R, "Zi":Zi, "cov":cov, "carry":carry}. Defaults to
            False.
        mem_splits : int
            Split the array in `mem_splits` segments to save memory.

        Returns
        -------
        out : array, shape=(n_channels, n_samples)
            Filtered data.

        """
        # extract the data
        X = raw.get_data(picks=picks)

        # add lookahead padding at the end
        lookahead_samples = int(self.sfreq * lookahead)
        X = np.concatenate([X,
                            np.zeros([X.shape[0], lookahead_samples])],
                          axis=1)

        # apply ASR
        X = asr_process(X, self.sfreq, self.M, self.T, self.win_len,
                        lookahead, stepsize, maxdims, (self.A, self.B),
                        self.R, self.Zi, self.cov, self.carry,
                        return_states, self.method, mem_splits)

        # remove lookahead portion from start
        X = X[:, lookahead_samples:]

        # Return a modifier raw instance
        raw = raw.copy()
        raw.apply_function(lambda x: X, picks=picks,
                           channel_wise=False)
        return raw


def asr_calibrate(X, sfreq, cutoff=20, blocksize=100, win_len=0.5,
                  win_overlap=0.66, max_dropout_fraction=0.1,
                  min_clean_fraction=0.25, ab=None, method='euclid'):
    """Calibration function for the Artifact Subspace Reconstruction method.

    This function can be used if you inted to apply ASR to a simple numpy
    array instead of a mne.io.Raw object. It is equivalent to the MATLAB
    implementation of asr_calibrate (except for some small differences
    introduced by solvers for the eigenspace functions etc).

    The input to this data is a multi-channel time series of calibration data.
    In typical uses the calibration data is clean resting EEG data of ca. 1
    minute duration (can also be longer). One can also use on-task data if the
    fraction of artifact content is below the breakdown point of the robust
    statistics used for estimation (50% theoretical, ~30% practical). If the
    data has a proportion of more than 30-50% artifacts then bad time windows
    should be removed beforehand. This data is used to estimate the thresholds
    that are used by the ASR processing function to identify and remove
    artifact components.

    The calibration data must have been recorded for the same cap design from
    which data for cleanup will be recorded, and ideally should be from the
    same session and same subject, but it is possible to reuse the calibration
    data from a previous session and montage to the extent that the cap is
    placed in the same location (where loss in accuracy is more or less
    proportional to the mismatch in cap placement).

    The calibration data should have been high-pass filtered (for example at
    0.5Hz or 1Hz using a Butterworth IIR filter).

    Parameters
    ----------
    X : array, shape=(n_channels, n_samples)
        *zero-mean* (e.g., high-pass filtered) and reasonably clean EEG of not
        much less than 30 seconds (this method is typically used with 1 minute
        or more).
    sfreq : float
        Sampling rate of the data, in Hz.
    cutoff: float
        Standard deviation cutoff for rejection. X portions whose variance
        is larger than this threshold relative to the calibration data are
        considered missing data and will be removed. Defaults to 20
        (In EEGLab's `clean_rawdata` the original threshold was set to 5, but
        it is widely recommended to use a value higher than 20).
    blocksize : int
        Block size for calculating the robust data covariance and thresholds,
        in samples; allows to reduce the memory and time requirements of the
        robust estimators by this factor (down to n_chans x n_chans x
        n_samples x 16 / blocksize bytes) (default=100).
    win_len : float
        Window length that is used to check the data for artifact content.
        This is ideally as long as the expected time scale of the artifacts
        but short enough to allow for several 1000 windows to compute
        statistics over (default=0.5).
    win_overlap : float
        Window overlap fraction. The fraction of two successive windows that
        overlaps. Higher overlap ensures that fewer artifact portions are
        going to be missed, but is slower (default=0.66).
    max_dropout_fraction : float
        Maximum fraction of windows that can be subject to signal dropouts
        (e.g., sensor unplugged), used for threshold estimation (default=0.1).
    min_clean_fraction : float
        Minimum fraction of windows that need to be clean, used for threshold
        estimation (default=0.25).
    ab : 2-tuple | None
        Coefficients (A, B) of an IIR filter that is used to shape the
        spectrum of the signal when calculating artifact statistics. The
        output signal does not go through this filter. This is an optional way
        to tune the sensitivity of the algorithm to each frequency component
        of the signal. The default filter is less sensitive at alpha and beta
        frequencies and more sensitive at delta (blinks) and gamma (muscle)
        frequencies. Defaults to None.
    method : {'euclid', 'riemann'}
        Metric to compute the covariance matrix average. For now, only
        euclidean ASR is supported.

    Returns
    -------
    M : array
        Mixing matrix.
    T : array
        Threshold matrix.

    """
    if method == "riemann":
        warnings.warn("Riemannian ASR is not yet supported. Switching back to"
                      " Euclidean ASR.")
        method == "euclid"

    logging.debug('[ASR] Calibrating...')

    # set number of channels and number of samples
    [nc, ns] = X.shape

    # filter the data
    X, _zf = yulewalk_filter(X, sfreq, ab=ab)

    # window length for calculating thresholds
    N = int(np.round(win_len * sfreq))

    # get block covariances
    U = block_covariance(X, window=blocksize)

    # get geometric median for each block
    # Note: riemann mode is not yet supported, else this could be:
    # Uavg = pyriemann.utils.mean_covariance(U, metric='riemann')
    Uavg = geometric_median(U.reshape((-1, nc * nc)) / blocksize)
    Uavg = Uavg.reshape((nc, nc))

    # get the mixing matrix M
    M = linalg.sqrtm(np.real(Uavg))

    # sort the get the sorted eigenvecotors/eigenvalues
    # riemann is not yet supported, else this could be PGA/nonlinear eigenvs
    D, Vtmp = linalg.eigh(M)
    V = Vtmp[:, np.argsort(D)]  # I think numpy sorts them automatically

    # get the threshold matrix T
    x = np.abs(np.dot(V.T, X))
    offsets = np.int_(np.arange(0, ns - N, np.round(N * (1 - win_overlap))))

    # go through all the channels and fit the EEG distribution
    mu = np.zeros(nc)
    sig = np.zeros(nc)
    for ichan in reversed(range(nc)):
        rms = x[ichan, :] ** 2
        Y = []
        for o in offsets:
            Y.append(np.sqrt(np.sum(rms[o:o + N]) / N))
        mu[ichan], sig[ichan], alpha, beta = fit_eeg_distribution(
            Y, min_clean_fraction, max_dropout_fraction)
    T = np.dot(np.diag(mu + cutoff * sig), V.T)

    logging.debug('[ASR] Calibration done.')
    return M, T


def asr_process(data, sfreq, M, T, windowlen=1, lookahead=0.001, stepsize=32,
                maxdims=0.66, ab=None, R=None, Zi=None, cov=None, carry=None,
                return_states=False, method="euclid", mem_splits=3):

    """Apply the Artifact Subspace Reconstruction method to a data array.

    This function is used to clean multi-channel signal using the ASR method.
    The required inputs are the data matrix and the sampling rate of the data.

    `asr_process` can be used if you inted to apply ASR to a simple numpy
    array instead of a mne.io.Raw object. It is equivalent to the MATLAB
    implementation of `asr_process` (except for some small differences
    introduced by solvers for the eigenspace functions etc).

    Parameters
    ----------
    data : array, shape=(n_channels, n_samples)
        Raw data.
    sfreq : float
        The sampling rate of the data.
    M : array, shape=(n_channels, n_channels)
        The Mixing matrix (as fitted with asr_calibrate).
    T : array, shape=(n_channels, n_channels)
        The Threshold matrix (as fitted with asr_calibrate).
    windowlen : float
        Window length that is used to check the data for artifact content.
        This is ideally as long as the expected time scale of the artifacts
        but short enough to allow for several 1000 windows to compute
        statistics over (default=0.5).
    lookahead:
        Amount of look-ahead that the algorithm should use. Since the
        processing is causal, the output signal will be delayed by this
        amount. This value is in seconds and should be between 0 (no
        lookahead) and WindowLength/2 (optimal lookahead). The recommended
        value is WindowLength/2. Default: 0.25
    stepsize:
        The steps in which the algorithm will be updated. The larger this is,
        the faster the algorithm will be. The value must not be larger than
        WindowLength * SamplingRate. The minimum value is 1 (update for every
        sample) while a good value would be sfreq//3. Note that an update
        is always performed also on the first and last sample of the data
        chunk. Default: 32
    max_dims : float, int
        Maximum dimensionality of artifacts to remove. This parameter
        denotes the maximum number of dimensions which can be removed from
        each segment. If larger than 1, `int(max_dims)` will denote the
        maximum number of dimensions removed from the data. If smaller than 1,
        `max_dims` describes a fraction of total dimensions. Defaults to 0.66.
    ab : 2-tuple | None
        Coefficients (A, B) of an IIR filter that is used to shape the
        spectrum of the signal when calculating artifact statistics. The
        output signal does not go through this filter. This is an optional way
        to tune the sensitivity of the algorithm to each frequency component
        of the signal. The default filter is less sensitive at alpha and beta
        frequencies and more sensitive at delta (blinks) and gamma (muscle)
        frequencies. Defaults to None.
    R : array, shape=(n_channels, n_channels)
        Previous reconstruction matrix. Defaults to None.
    Zi : array
        Previous filter conditions. Defaults to None.
    cov : array, shape=([n_trials, ]n_channels, n_channels) | None
        Covariance. If None (default), then it is computed from ``X_filt``.
        If a 3D array is provided, the average covariance is computed from
        all the elements in it. Defaults to None.
    carry :
        Initial portion of the data that will be added to the current data.
        If None, data will be interpolated. Defaults to None.
    return_states : bool
        If True, returns a dict including the updated states {"M":M, "T":T,
        "R":R, "Zi":Zi, "cov":cov, "carry":carry}. Defaults to False.
    method : {'euclid', 'riemann'}
        Metric to compute the covariance matrix average. Currently, only
        euclidean ASR is supported.
    mem_splits : int
        Split the array in `mem_splits` segments to save memory.


    Returns
    -------
    clean : array, shape=(n_channels, n_samples)
        Clean data.
    state : dict
        Output ASR parameters {"M":M, "T":T, "R":R, "Zi":Zi, "cov":cov,
        "carry":carry}.

    """
    if method == "riemann":
        warnings.warn("Riemannian ASR is not yet supported. Switching back to"
                      " Euclidean ASR.")
        method == "euclid"

    # calculate the the actual max dims based on the fraction parameter
    if maxdims < 1:
        maxdims = np.round(len(data) * maxdims)

    # set initial filter conditions of none was passed
    if Zi is None:
        _, Zi = yulewalk_filter(data, ab=ab, sfreq=sfreq,
                                zi=np.ones([len(data), 8]))

    # set the number of channels
    C, S = data.shape

    # set the number of windows
    N = np.round(windowlen * sfreq).astype(int)
    P = np.round(lookahead * sfreq).astype(int)

    # interpolate a portion of the data if no buffer was given
    if carry is None:
        carry = np.tile(2 * data[:, 0],
                        (P, 1)).T - data[:, np.mod(np.arange(P, 0, -1), S)]
    data = np.concatenate([carry, data], axis=-1)

    # splits = np.ceil(C*C*S*8*8 + C*C*8*s/stepsize + C*S*8*2 + S*8*5)...
    splits = mem_splits  # TODO: use this for parallelization MAKE IT A PARAM FIRST

    # loop over smaller segments of the data (for memory purposes)
    last_trivial = False
    last_R = None
    for i in range(splits):

        # set the current range
        i_range = np.arange(i * S // splits,
                            np.min([(i + 1) * S // splits, S]),
                            dtype=int)

        # filter the current window with yule-walker
        X, Zi = yulewalk_filter(data[:, i_range + P], sfreq=sfreq,
                                zi=Zi, ab=ab, axis=-1)

        # compute a moving average covariance
        Xcov, cov = \
            ma_filter(N,
                      np.reshape(np.multiply(np.reshape(X, (1, C, -1)),
                                             np.reshape(X, (C, 1, -1))),
                                 (C * C, -1)), cov)

        # set indices at which we update the signal
        update_at = np.arange(stepsize,
                              Xcov.shape[-1] + stepsize - 2,
                              stepsize)
        update_at = np.minimum(update_at, Xcov.shape[-1]) - 1

        # set the previous reconstruction matrix if none was assigned
        if last_R is None:
            update_at = np.concatenate([[0], update_at])
            last_R = np.eye(C)

        Xcov = np.reshape(Xcov[:, update_at], (C, C, -1))

        # loop through the updating intervals
        last_n = 0
        for j in range(len(update_at) - 1):

            # get the eigenvectors/values.For method 'riemann', this should
            # be replaced with PGA/ nonlinear eigenvalues
            D, V = np.linalg.eigh(Xcov[:, :, j])

            # determine which components to keep
            keep = np.logical_or(D < np.sum((T @ V)**2, axis=0),
                                 np.arange(C) + 1 < (C - maxdims))
            trivial = np.all(keep)

            # set the reconstruction matrix (ie. reconstructing artifact
            # components using the mixing matrix)
            if not trivial:
                inv = pinv(np.multiply(keep[:, np.newaxis], V.T @ M))
                R = np.real(M @ inv @ V.T)
            else:
                R = np.eye(C)

            # apply the reconstruction
            n = update_at[j] + 1
            if (not trivial) or (not last_trivial):

                subrange = i_range[np.arange(last_n, n)]

                # generate a cosine signal
                blend_x = np.pi * np.arange(1, n - last_n + 1) / (n - last_n)
                blend = (1 - np.cos(blend_x)) / 2

                # use cosine blending to replace data with reconstructed data
                tmp_data = data[:, subrange]
                data[:, subrange] = np.multiply(blend, R @ tmp_data) + \
                                    np.multiply(1 - blend, last_R @ tmp_data) # noqa

            # set the parameters for the next iteration
            last_n, last_R, last_trivial = n, R, trivial

    # assign a new lookahead portion
    carry = np.concatenate([carry, data[:, -P:]])
    carry = carry[:, -P:]

    if return_states:
        return data[:, :-P], {"M": M, "T": T, "R": R, "Zi": Zi,
                              "cov": cov, "carry": carry}
    else:
        return data[:, :-P]


def clean_windows(X, sfreq, max_bad_chans=0.2, zthresholds=[-3.5, 5],
                  win_len=1, win_overlap=0.66, min_clean_fraction=0.25,
                  max_dropout_fraction=0.1):
    """Remove periods with abnormally high-power content from continuous data.

    This function cuts segments from the data which contain high-power
    artifacts. Specifically, only windows are retained which have less than a
    certain fraction of "bad" channels, where a channel is bad in a window if
    its power is above or below a given upper/lower threshold (in standard
    deviations from a robust estimate of the EEG power distribution in the
    channel).

    Parameters
    ----------
    X : array, shape=(n_channels, n_samples)
        Continuous data set, assumed to be appropriately high-passed (e.g. >
        1Hz or 0.5Hz - 2.0Hz transition band)
    max_bad_chans : float
        The maximum number or fraction of bad channels that a retained window
        may still contain (more than this and it is removed). Reasonable range
        is 0.05 (very clean output) to 0.3 (very lax cleaning of only coarse
        artifacts) (default=0.2).
    zthresholds : 2-tuple
        The minimum and maximum standard deviations within which the power of
        a channel must lie (relative to a robust estimate of the clean EEG
        power distribution in the channel) for it to be considered "not bad".
        (default=[-3.5, 5]).

    The following are detail parameters that usually do not have to be tuned.
    If you can't get the function to do what you want, you might consider
    adapting these to your data.

    win_len : float
        Window length that is used to check the data for artifact content.
        This is ideally as long as the expected time scale of the artifacts
        but not shorter than half a cycle of the high-pass filter that was
        used. (Default: 1).
    win_overlap : float
        Window overlap fraction. The fraction of two successive windows that
        overlaps. Higher overlap ensures that fewer artifact portions are
        going to be missed, but is slower (default=0.66).
    min_clean_fraction : float
        Minimum fraction that needs to be clean. This is the minimum fraction
        of time windows that need to contain essentially uncontaminated EEG.
        (default=0.25)
    max_dropout_fraction : float
        Maximum fraction that can have dropouts. This is the maximum fraction
        of time windows that may have arbitrarily low amplitude (e.g., due to
        the sensors being unplugged) (default=0.1).

    Returns
    -------
    clean : array, shape=(n_channels, n_samples)
        Dataset with bad time periods removed.
    sample_mask : boolean array, shape=(1, n_samples)
        Mask of retained samples (logical array).

    """
    assert 0 < max_bad_chans < 1, "max_bad_chans must be a fraction !"

    # set internal variables
    truncate_quant = [0.0220, 0.6000]
    step_sizes = [0.01, 0.01]
    shape_range = np.arange(1.7, 3.5, 0.15)
    max_bad_chans = np.round(X.shape[0] * max_bad_chans)

    # set data indices
    [nc, ns] = X.shape
    N = int(win_len * sfreq)
    offsets = np.int_(np.round(np.arange(0, ns - N, (N * (1 - win_overlap)))))
    logging.debug('[ASR] Determining channel-wise rejection thresholds')

    wz = np.zeros((nc, len(offsets)))
    for ichan in range(nc):

        # compute root mean squared amplitude
        x = X[ichan, :] ** 2
        Y = np.array([np.sqrt(np.sum(x[o:o + N]) / N) for o in offsets])

        # fit a distribution to the clean EEG part
        mu, sig, alpha, beta = fit_eeg_distribution(
            Y, min_clean_fraction, max_dropout_fraction, truncate_quant,
            step_sizes, shape_range)
        # calculate z scores
        wz[ichan] = (Y - mu) / sig

    # sort z scores into quantiles
    wz[np.isnan(wz)] = np.inf  # Nan to inf
    swz = np.sort(wz, axis=0)

    # determine which windows to remove
    if np.max(zthresholds) > 0:
        mask1 = swz[-(int(max_bad_chans) + 1), :] > np.max(zthresholds)
    if np.min(zthresholds) < 0:
        mask2 = (swz[1 + int(max_bad_chans - 1), :] < np.min(zthresholds))

    # combine the two thresholds
    remove_mask = np.logical_or.reduce((mask1, mask2))
    removed_wins = np.where(remove_mask)

    # reconstruct the samples to remove
    sample_maskidx = []
    for i in range(len(removed_wins[0])):
        if i == 0:
            sample_maskidx = np.arange(
                offsets[removed_wins[0][i]], offsets[removed_wins[0][i]] + N)
        else:
            sample_maskidx = np.vstack((
                sample_maskidx,
                np.arange(offsets[removed_wins[0][i]],
                          offsets[removed_wins[0][i]] + N)
            ))

    # delete the bad chunks from the data
    sample_mask2remove = np.unique(sample_maskidx)
    if sample_mask2remove.size:
        clean = np.delete(X, sample_mask2remove, 1)
        sample_mask = np.ones((1, ns), dtype=bool)
        sample_mask[0, sample_mask2remove] = False
    else:
        sample_mask = np.ones((1, ns), dtype=bool)
        clean = X
        print('Try calibrating ASR with cleaner data.')

    return clean, sample_mask

In [15]:
# ==============================================================================
# CELL 14: Baseline Method 2 - Artifact Subspace Reconstruction
# ==============================================================================

def denoiseWithArtifaceSuspaceReconstructionwithASROffline(rawDATA, sampleFreq, n_channels, cutOffASR,
                                       window_size_sec, channelNames, precleaned=True, verbose=False):

  #rawDATA = dataMNEObject.get_data()
  lenOfRawData = len (rawDATA[0])
  maxWindows = int((len(rawDATA[0])/sampleFreq) / window_size_sec)  #retrieve the max EEG win of window_size_sec to clean individually
  fullCleanEEGData = np.zeros((n_channels, lenOfRawData), dtype = 'float')  #container of clearn data
  lengthToplot = int(lenOfRawData/6)

  # (optional) make sure your asr is only fitted to clean parts of the data (this is taken from the whole data)
  calibrationData = rawDATA #the whole data is used, since it is offline, and this is possible

  maxAmp = np.max(calibrationData.max(axis=1))
  minAmp  = np.min(calibrationData.min(axis=1))

  if (verbose):
    print ("Calibration data shape before cleaning: {}".format(calibrationData.shape))
    plt.plot(calibrationData[:,0:lengthToplot])
    plt.ylim(bottom=minAmp)
    plt.ylim(top=maxAmp)
    plt.show()


  if (precleaned):
    min_clean_fraction = 0.25
    max_dropout_fraction = 0.1
    calibrationDataCleaned, _ = clean_windows(calibrationData, sampleFreq, max_bad_chans=0.05, win_len=window_size_sec, min_clean_fraction=min_clean_fraction, max_dropout_fraction=max_dropout_fraction)

  if (verbose):
    print ("Calibration data shape after cleaning: {}".format(calibrationData.shape))
    plt.plot(calibrationDataCleaned[:,0:lengthToplot])
    plt.ylim(bottom=minAmp)
    plt.ylim(top=maxAmp)
    plt.show()

  # fit the ASR
  M, T = asr_calibrate(calibrationDataCleaned, sampleFreq, cutoff=cutOffASR, win_len=window_size_sec)

  if (verbose):
    print ("Full Raw data before ASR: {}".format(rawDATA.shape))
    plt.plot(rawDATA[:,0:lengthToplot])
    plt.ylim(bottom=minAmp)
    plt.ylim(top=maxAmp)
    plt.show()

  # clean data
  clean_array = asr_process(rawDATA, sampleFreq, M, T, windowlen=window_size_sec)

  if (verbose):
    print ("Full Raw data after ASR: {}".format(clean_array.shape))
    plt.plot(clean_array[:,0:lengthToplot])
    plt.ylim(bottom=minAmp)
    plt.ylim(top=maxAmp)
    plt.show()

  return clean_array

def denoiseWithArtifaceSuspaceReconstructionwithASRRealTimeLogic(rawDATA, sampleFreq, calibLengthInPoints, n_channels,
                          cutOffASR, window_size_sec, channelNames, precleaned=False, verbose=False):

  #rawDATA = dataMNEObject.get_data()
  maxWindows = int((len(rawDATA[0])/sampleFreq) / window_size_sec)  #retrieve the max EEG win of window_size_sec to clean individually
  fullCleanEEGData = np.zeros((n_channels, len (rawDATA[0])), dtype = 'float')  #container of clean data

  #create a numpy array of EEG data from the MNE raw object
  calibrationData = rawDATA[:,0:calibLengthInPoints]

  # (optional) make sure  asr is only fitted to clean parts of the calibration data
  if (precleaned):
    min_clean_fraction = 0.35
    max_dropout_fraction = 0.05
    pre_cleaned, _ = clean_windows(calibrationData, sampleFreq, max_bad_chans=0.1, win_len=window_size_sec, min_clean_fraction=min_clean_fraction, max_dropout_fraction=max_dropout_fraction)
    calibrationData = pre_cleaned

  # fit the asr only with the calibration data
  #bs = len(calibrationData[0]) #the length in points of the calibration interval
  M, T = asr_calibrate(calibrationData, sampleFreq, cutoff=cutOffASR, win_len=window_size_sec, min_clean_fraction=min_clean_fraction, max_dropout_fraction=max_dropout_fraction)

  # ---- Real-time processing loop ----
  for win in range(maxWindows):  #loop through all the EEG windows
    windowData = np.zeros((n_channels, int(sampleFreq)), dtype = 'float') #container of single window
    start = int(win*sampleFreq)
    stop = int((win+1)*sampleFreq)

    if (win == (maxWindows-1)):
      windowData = rawDATA[:, start:] #last window
    else:
      windowData = rawDATA[:, start:stop]

    #clean the current window
    clean_win = asr_process(windowData, sampleFreq, M, T, windowlen=window_size_sec)

    fullCleanEEGData[:, start:stop] = clean_win

  return fullCleanEEGData

In [16]:
# ==========================================
# CELL 15: Quality Metrics Evaluator
# ==========================================
#  SNR-Diff / PSNR-Diff for real data (without ground truth) now yield deterministic values.
#  Signal = Power of the trial-averaged ERP waveform within the post-stimulus window
#  Noise  = Noise variance of the pre-stimulus baseline period (1/n) * sum_n delta_n^2
#  Both metrics require only a single signal + event markers, without needing a clean reference.
#
#Implemented metrics:
#  SNR-Diff   : Signal-to-Noise Ratio difference
#  PSNR-Diff  : Peak Signal-to-Noise Ratio difference
#  SAR        : Signal-to-Artifact Ratio
#  RMSE       : Root Mean Square Error
#  Corr       : Pearson correlation coefficient
#  MI         : Mutual Information
#  BPPE       : Band Power Preservation Error (difference in relative band power)
#  MSC        : Magnitude-Squared Coherence
# ==========================================

import warnings
import numpy as np
import pandas as pd
from scipy.stats import pearsonr
from scipy.signal import welch, coherence
from sklearn.metrics import mutual_info_score

warnings.filterwarnings("ignore")

_EPS = 1e-12
_trapezoid = getattr(np, "trapezoid", np.trapz)

# Half-open intervals to prevent boundary frequencies (4/8/13/30 Hz) from being counted twice
CANONICAL_BANDS = {
    "Delta": (0.5, 4.0),
    "Theta": (4.0, 8.0),
    "Alpha": (8.0, 13.0),
    "Beta":  (13.0, 30.0),
    "Gamma": (30.0, 50.0),
}

QUALITY_METRICS = (
    ["SNR-Diff", "PSNR-Diff", "RMSE", "Corr", "MI"]
    + [f"BPPE_{b}" for b in CANONICAL_BANDS]
    + [f"MSC_{b}" for b in CANONICAL_BANDS]
)

# Table 4.5: lower = better
METRICS_LOWER_BETTER = [
    "PSNR-Diff", "RMSE",
    "BPPE_Delta", "BPPE_Theta", "BPPE_Alpha", "BPPE_Beta", "BPPE_Gamma",
    "MSC_Delta", "MSC_Theta", "MSC_Gamma",
]

NON_METRIC_COLS = ["Subject", "Channel", "Bs", "Method",
                   "Avg_Latency_ms", "EvalMode", "N_Trials"]


def _db_ratio(numerator, denominator, cap=100.0):
    """Safe 10*log10(num/den); den->0 indicates perfect reconstruction, returning +cap instead of 0.0."""
    if not np.isfinite(numerator) or not np.isfinite(denominator):
        return np.nan
    if numerator <= _EPS:
        return -float(cap)
    if denominator <= _EPS:
        return float(cap)
    return float(np.clip(10.0 * np.log10(numerator / denominator), -cap, cap))


def _as_1d(x):
    return np.asarray(x, dtype=np.float64).ravel()


class OnEEGWaveLAD_MetricsEvaluator:
    """
    mode="simulated" : reference = clean ground truth, raw = artifact-contaminated input (required)
    mode="real"      : reference = original recording; SNR/PSNR evaluated via ERP baseline method or VEOG-SAR

    Priority for calculating SNR-Diff / PSNR-Diff in 'real' mode:
      1) event_samples provided -> ERP baseline method (Recommended, Eq 2.11-2.15)
      2) artefact_ref provided  -> SAR improvement based on VEOG reference (Eq 2.16), PSNR-Diff = NaN
      3) Neither provided       -> NaN

    This metric strictly avoids using "removed energy" to deduce SNR, as such an approach
    unfairly assigns the highest score to "do-nothing" methods.
    """

    def __init__(self, fs=1024.0, bands=None, mode="real",
                 welch_seconds=2.0, mi_bins=50, db_cap=100.0,
                 baseline_ms=(-200.0, 0.0), erp_ms=(0.0, 500.0),
                 min_trials=10):
        self.fs = float(fs)
        self.bands = dict(bands) if bands is not None else dict(CANONICAL_BANDS)
        self.mode = mode
        self.mi_bins = int(mi_bins)
        self.db_cap = float(db_cap)
        self.min_trials = int(min_trials)

        # welch_seconds=2 -> 0.5 Hz resolution, required to cover the lower bound of the Delta band (0.5 Hz)
        self._nperseg = int(round(welch_seconds * self.fs))

        ms2s = lambda t: int(round(t * self.fs / 1000.0))
        self._b0, self._b1 = ms2s(baseline_ms[0]), ms2s(baseline_ms[1])
        self._e0, self._e1 = ms2s(erp_ms[0]), ms2s(erp_ms[1])

    # ==========================================================================
    # 2.5.1  Signal-to-noise-type
    # ==========================================================================
    def _epoch(self, x, event_samples):
        """Epoch the signal into baseline and ERP segments based on events, applying trial-by-trial baseline correction."""
        n = len(x)
        base, erp = [], []
        for s in np.asarray(event_samples, dtype=int):
            ib0, ib1 = s + self._b0, s + self._b1
            ie0, ie1 = s + self._e0, s + self._e1
            if ib0 < 0 or ie1 > n or ib1 <= ib0 or ie1 <= ie0:
                continue
            seg_b = x[ib0:ib1]
            offset = float(np.mean(seg_b))          # Trial-by-trial baseline correction
            base.append(seg_b - offset)
            erp.append(x[ie0:ie1] - offset)
        if len(base) < self.min_trials:
            return None, None
        return np.asarray(base), np.asarray(erp)

    def calc_erp_snr_psnr(self, signal, event_samples):
        """
        Reference-free SNR / PSNR, based on Eq. (2.12) and Eq. (2.15).

            SNR  = 10*log10( mean(r_s^2) / (1/n) sum_n delta_n^2 )
            PSNR = 10*log10( max(r_s^2)  / (1/n) sum_n delta_n^2 )

        r_s : Trial-averaged ERP waveform (post-stimulus window)
        delta_n^2 : Variance of the baseline segment for each trial, averaged across trials as per Eq. (2.12)

        Returns (snr_db, psnr_db, n_trials)
        """
        base, erp = self._epoch(_as_1d(signal), event_samples)
        if base is None:
            return np.nan, np.nan, 0

        # Noise: Average of baseline variances across trials (denominator of Eq. 2.12)
        noise_var = float(np.mean(np.var(base, axis=1)))

        # Signal: Trial-averaged ERP waveform
        erp_avg = np.mean(erp, axis=0)
        p_mean = float(np.mean(erp_avg ** 2))
        p_peak = float(np.max(erp_avg ** 2))

        snr = _db_ratio(p_mean, noise_var, self.db_cap)
        psnr = _db_ratio(p_peak, noise_var, self.db_cap)
        return snr, psnr, int(base.shape[0])

    def calc_erp_snr_psnr_diff(self, raw_signal, denoised, event_samples):
        """
        SNR-Diff  = SNR(denoised) - SNR(raw)      Table 4.5: higher = better
        PSNR-Diff = PSNR(raw) - PSNR(denoised)    Table 4.5: lower  = better

        Directionality note: The numerator derives from the ERP waveform, while the denominator comes from baseline noise.
          - Do nothing -> Baseline retains artifacts, yielding a large denominator -> Low SNR
          - Over-smoothing -> ERP is attenuated, yielding a small numerator -> Low SNR
        Both extremes are penalized, a crucial property missing from the "removed energy" formulations.
        """
        snr_raw, psnr_raw, n_tr = self.calc_erp_snr_psnr(raw_signal, event_samples)
        snr_den, psnr_den, _ = self.calc_erp_snr_psnr(denoised, event_samples)
        if not np.isfinite(snr_raw) or not np.isfinite(snr_den):
            return np.nan, np.nan, n_tr
        return float(snr_den - snr_raw), float(psnr_raw - psnr_den), n_tr

    def calc_snr_psnr_diff_simulated(self, reference, denoised, raw):
        """Simulated mode: 'reference' is the clean ground truth, 'raw' is the artifact-contaminated input."""
        if raw is None:
            return np.nan, np.nan
        mse_raw = float(np.mean((reference - raw) ** 2))
        mse_den = float(np.mean((reference - denoised) ** 2))
        var_ref = float(np.var(reference))
        max_val = float(np.max(np.abs(reference)))

        snr_diff = (_db_ratio(var_ref, mse_den, self.db_cap)
                    - _db_ratio(var_ref, mse_raw, self.db_cap))
        psnr_diff = (_db_ratio(max_val ** 2, mse_raw, self.db_cap)
                     - _db_ratio(max_val ** 2, mse_den, self.db_cap))
        return float(snr_diff), float(psnr_diff)

    def calc_sar_improvement(self, raw_signal, denoised, artefact_ref):
        """
        Operationalization of Eq. (2.16): Estimates artifact power via the projection of an
        external artifact reference (e.g., VEOG/EMG) onto the target channel.
            beta  = <x, a> / <a, a>;  P_art = beta^2 * P(a);  P_eeg = P(x) - P_art
            SAR   = 10*log10(P_eeg / P_art)
        Returns SAR(denoised) - SAR(raw); higher is better.
        Artifacts are defined using an independently measured external channel, not derived
        from the denoising method's own output.
        """
        if artefact_ref is None:
            return np.nan
        a = _as_1d(artefact_ref)
        n = min(len(a), len(raw_signal), len(denoised))
        a, x_raw, x_den = a[:n], raw_signal[:n], denoised[:n]
        a = a - np.mean(a)
        pa = float(np.dot(a, a))
        if pa <= _EPS:
            return np.nan

        def _sar(x):
            x = x - np.mean(x)
            beta = float(np.dot(x, a)) / pa
            p_art = (beta ** 2) * pa
            p_eeg = max(float(np.dot(x, x)) - p_art, 0.0)
            return _db_ratio(p_eeg, p_art, self.db_cap)

        return float(_sar(x_den) - _sar(x_raw))

    # Amplitude-based similarity
    def calc_rmse(self, reference, denoised):
        """Eq. (2.18)"""
        return float(np.sqrt(np.mean((reference - denoised) ** 2)))

    def calc_correlation(self, reference, denoised):
        """Eq. (2.20)"""
        if np.std(reference) < _EPS or np.std(denoised) < _EPS:
            return 0.0
        c, _ = pearsonr(reference, denoised)
        return float(c) if np.isfinite(c) else 0.0


    # Information-theoretic
    def calc_mutual_information(self, reference, denoised, edges=None):
        """
        Eq. (2.21), in bits.
        Bin edges are fixed based on the reference (or passed as shared boundaries).
        Otherwise, method-specific dynamic ranges would dictate bin widths, rendering
        MI values incomparable across methods.
        """
        if edges is None:
            lo, hi = float(np.min(reference)), float(np.max(reference))
            if hi - lo < _EPS:
                return 0.0
            pad = 0.05 * (hi - lo)
            edges = np.linspace(lo - pad, hi + pad, self.mi_bins + 1)
        x = np.clip(reference, edges[0], edges[-1])
        y = np.clip(denoised, edges[0], edges[-1])
        c_xy, _, _ = np.histogram2d(x, y, bins=[edges, edges])
        if c_xy.sum() <= 0:
            return 0.0
        return float(mutual_info_score(None, None, contingency=c_xy) / np.log(2.0))

    # Spectral preservation
    def calc_spectral_metrics(self, reference, denoised):
        """
        BPPE (Sec 2.5.4.2): Difference in relative band power p_b = P_b / sum_k P_k, E_b = |p_o - p_d|
          -> Invariant to global amplitude scaling (e.g., average re-referencing in 4.3.4.1 alters channel scale)
          -> Naturally bounded within [0, 1]
        MSC (Eq. 2.23): Mean magnitude squared coherence within each frequency band
        """
        n = min(len(reference), len(denoised))
        nperseg = int(min(self._nperseg, n))
        if nperseg < 8:
            nan_d = {b: np.nan for b in self.bands}
            return nan_d, dict(nan_d)

        f, psd_ref = welch(reference, fs=self.fs, nperseg=nperseg)
        _, psd_den = welch(denoised, fs=self.fs, nperseg=nperseg)
        f_coh, coh = coherence(reference, denoised, fs=self.fs, nperseg=nperseg)

        degenerate = np.std(denoised) < _EPS

        p_ref, p_den, msc = {}, {}, {}
        for band, (fmin, fmax) in self.bands.items():
            idx = (f >= fmin) & (f < fmax)
            idx_c = (f_coh >= fmin) & (f_coh < fmax)
            p_ref[band] = float(_trapezoid(psd_ref[idx], f[idx])) if np.any(idx) else 0.0
            p_den[band] = float(_trapezoid(psd_den[idx], f[idx])) if np.any(idx) else 0.0
            if degenerate or not np.any(idx_c):
                msc[band] = 0.0
            else:
                v = np.nanmean(coh[idx_c])
                msc[band] = float(v) if np.isfinite(v) else 0.0

        tot_ref, tot_den = sum(p_ref.values()), sum(p_den.values())
        bppe = {}
        for band in self.bands:
            if tot_ref <= _EPS:
                bppe[band] = np.nan
            elif tot_den <= _EPS:
                bppe[band] = float(p_ref[band] / tot_ref)
            else:
                bppe[band] = float(abs(p_ref[band] / tot_ref - p_den[band] / tot_den))
        return bppe, msc

    # Public Interface
    def evaluate_channel(self, reference, denoised, raw=None,
                         event_samples=None, artefact_ref=None,
                         mode=None, mi_edges=None):
        """
        Positional arguments for backward compatibility: evaluate_channel(orig, denoised, noisy)

        reference    : Clean ground truth for 'simulated' mode; original recording for 'real' mode
        denoised     : Denoised output
        raw          : Artifact-contaminated input for 'simulated' mode; defaults to reference if None in 'real' mode
        event_samples: Sample indices of stimulus events (strongly recommended in 'real' mode)
        artefact_ref : External VEOG/EMG reference (secondary alternative for 'real' mode)
        """
        mode = mode or self.mode
        if mode not in ("real", "simulated"):
            raise ValueError("mode must be 'real' or 'simulated'")

        ref = _as_1d(reference)
        den = _as_1d(denoised)
        n = min(len(ref), len(den))
        ref, den = ref[:n], den[:n]
        raw_arr = _as_1d(raw)[:n] if raw is not None else None

        eval_mode, n_trials = mode, 0

        if mode == "simulated":
            if raw_arr is None:
                raise ValueError("mode='simulated' requires 'raw' (artifact-contaminated input)")
            snr_diff, psnr_diff = self.calc_snr_psnr_diff_simulated(ref, den, raw_arr)
            eval_mode = "simulated"
        else:
            base_signal = raw_arr if raw_arr is not None else ref
            if event_samples is not None and len(np.asarray(event_samples)) > 0:
                # Retain only events whose epochs fully reside within the valid signal length
                ev = np.asarray(event_samples, dtype=int)
                ev = ev[(ev + self._b0 >= 0) & (ev + self._e1 <= n)]
                snr_diff, psnr_diff, n_trials = self.calc_erp_snr_psnr_diff(
                    base_signal, den, ev)
                eval_mode = "real-erp"
            elif artefact_ref is not None:
                snr_diff = self.calc_sar_improvement(base_signal, den, artefact_ref)
                psnr_diff = np.nan
                eval_mode = "real-sar"
            else:
                snr_diff, psnr_diff = np.nan, np.nan
                eval_mode = "real-none"

        results = {
            "SNR-Diff": snr_diff,
            "PSNR-Diff": psnr_diff,
            "RMSE": self.calc_rmse(ref, den),
            "Corr": self.calc_correlation(ref, den),
            "MI": self.calc_mutual_information(ref, den, edges=mi_edges),
        }
        bppe, msc = self.calc_spectral_metrics(ref, den)
        for band in self.bands:
            results[f"BPPE_{band}"] = bppe[band]
            results[f"MSC_{band}"] = msc[band]

        results["EvalMode"] = eval_mode
        results["N_Trials"] = n_trials
        return results


print("[CELL 1 v3] Evaluator loaded.")
print(f"  metrics       : {len(QUALITY_METRICS)}")
print(f"  lower=better  : {METRICS_LOWER_BETTER}")
print("  Provide 'event_samples' in 'real' mode to obtain valid SNR-Diff / PSNR-Diff values")

[CELL 1 v3] Evaluator loaded.
  metrics       : 15
  lower=better  : ['PSNR-Diff', 'RMSE', 'BPPE_Delta', 'BPPE_Theta', 'BPPE_Alpha', 'BPPE_Beta', 'BPPE_Gamma', 'MSC_Delta', 'MSC_Theta', 'MSC_Gamma']
  Provide 'event_samples' in 'real' mode to obtain valid SNR-Diff / PSNR-Diff values


In [17]:
%%script false --no-raise-error
# ==========================================
# CELL 16: Batch Processing: onEEGwaveLAD Framework
# ==========================================
import os
import time
import mne
import numpy as np
import pandas as pd
from joblib import Parallel, delayed

#Global configuration
datasetName = "N170"
rootPath = "/content/drive/MyDrive/EEG_Project/"
montageName = "standard_1020"

EEG_30 = ["FP1","F3","F7","FC3","C3","C5","P3","P7","P9","PO7","PO3","O1","Oz","Pz","CPz",
          "FP2","Fz","F4","F8","FC4","FCz","Cz","C4","C6","P4","P8","P10","PO8","PO4","O2"]
EOG_CH = ["HEOG_left","HEOG_right","VEOG_lower"]

usedChannels = EEG_30 + EOG_CH
montageChannelNames = EEG_30
parametersOfDenoiser = {'RTWL': 1000}

START_SUBJECT = 1
END_SUBJECT = 30
TEST_MODE = False  #Set to True to test 1 channel only (warning: mathematically imprecise reference)


#Preprocessing helper functions
def fix_channel_types(raw):
    """
    Relabel EOG channels from 'eeg' to 'eog' to avoid contamination.
    """
    r = raw.copy()
    present = {c: 'eog' for c in EOG_CH if c in r.ch_names}
    if present:
        r.set_channel_types(present, verbose='ERROR')
    return r

def preprocess_1hz(raw_full):
    """
    Apply 1Hz highpass filter and select 30 EEG channels.
    Average referencing is deferred until after denoising.
    """
    r = fix_channel_types(raw_full).load_data()
    r.pick(EEG_30)
    r.filter(1., None, verbose='ERROR')
    return r

def avg_reref_flat(data):
    """
    Apply average reference with virtual flat channel to (n_ch, T) array.
    Each channel is subtracted by (sum of all channels)/(n_ch+1).
    """
    return data - data.sum(axis=0, keepdims=True) / (data.shape[0] + 1)


#Parallel single-channel denoising worker
def _denoise_one_channel(ch_name, bs, sig1d, window_samples, fs):
    """
    Denoise a single channel and return 1D denoised signal.
    Creates writable copy to avoid joblib read-only memmap conflicts.
    """
    sig1d = np.array(sig1d, dtype=np.float64)
    denoiser = OnEEGWaveLAD_Denoiser(n_channels=1, Bs=bs, IFS=512, Ta=0.55, Es=35)
    dwt = OnEEGWaveLAD_DWT(MW='sym4')
    out = np.zeros_like(sig1d)

    for s in range(0, len(sig1d), window_samples):
        e = s + window_samples
        if e > len(sig1d):
            break
        coeffs = dwt.decompose_window(sig1d[s:e].reshape(1, -1))
        out[s:e] = denoiser.process_window(coeffs, window_samples)[0]

    return ch_name, out


#Single-subject main controller
def evaluate_single_subject_parallel(subject_id, bs_list):
    print(f"\n{'='*50}\nProcessing Subject: sub-{subject_id}\n{'='*50}")
    try:
        raw_full = loadDataset(
            name=datasetName, subject=str(subject_id), montageName=montageName,
            usedChannels=usedChannels, rootPath=rootPath,
            montageChannelNames=montageChannelNames,
            parametersOfDenoiser=parametersOfDenoiser, verbose=False)
    except Exception as e:
        print(f"[WARNING] sub-{subject_id} loading failed, skipping: {e}")
        return []

    #Extract events (from pre-filtered object; filtering does not alter time axis)
    try:
        events, event_id = mne.events_from_annotations(raw_full, verbose=False)
        stim_codes = {v for k, v in event_id.items()
                      if str(k).strip().isdigit() and 1 <= int(str(k).strip()) <= 80}
        if stim_codes:
            events = events[np.isin(events[:, 2], list(stim_codes))]
        event_samples = events[:, 0].astype(int)
    except Exception as e:
        print(f"[WARNING] Event extraction failed ({e}).")
        event_samples = np.array([], dtype=int)
    print(f"[{subject_id}] extracted {len(event_samples)} events.")

    #1Hz highpass only, 30 channels without re-referencing
    r = preprocess_1hz(raw_full)
    fs = r.info['sfreq']
    ch_names = list(r.ch_names)
    data_np = r.get_data()

    windowing = OnEEGWaveLAD_Windowing(r, RTWL=parametersOfDenoiser['RTWL'])
    window_samples = windowing.window_samples
    eval_len = (data_np.shape[1] // window_samples) * window_samples

    #Reference = re-referenced (1Hz) original data, Bs-independent, computed once
    ref_reref = avg_reref_flat(data_np)

    target = ch_names[:1] if TEST_MODE else ch_names
    if TEST_MODE and len(target) < len(ch_names):
        print("[WARNING] TEST_MODE single channel cannot produce valid average reference; set TEST_MODE=False for full run")

    evaluator = OnEEGWaveLAD_MetricsEvaluator(fs=fs, mode="real")
    rows = []

    for bs in bs_list:
        den = Parallel(n_jobs=-1, verbose=10)(
            delayed(_denoise_one_channel)(
                ch, bs, data_np[ch_names.index(ch)].copy(), window_samples, fs
            ) for ch in target
        )

        D = np.zeros((len(target), eval_len))
        for ch, out in den:
            D[target.index(ch)] = out[:eval_len]

        #Apply average reference to denoised output
        D_reref = avg_reref_flat(D)

        for ci, ch in enumerate(target):
            ref_ch = ref_reref[ch_names.index(ch), :eval_len]
            m = evaluator.evaluate_channel(
                reference=ref_ch.copy(),
                denoised=D_reref[ci].copy(),
                raw=ref_ch.copy(),
                event_samples=event_samples, mode="real"
            )
            rows.append({"Subject": f"sub-{subject_id}", "Channel": ch, "Bs": bs, **m})

        print(f"  [sub-{subject_id}] Bs={bs:2d} done.")

    #Sort by channel topological order, then by Bs value
    rows.sort(key=lambda x: (target.index(x["Channel"]), x["Bs"]))

    return rows


#Main loop
if __name__ == "__main__":
    BS_LIST_TO_TEST = [1, 2, 4, 8, 16, 32, 64]
    valid_subjects = list(range(START_SUBJECT, END_SUBJECT + 1))

    csv_name = f"onEEGWaveLAD_Metrics_POST_REF_Sub_{START_SUBJECT}_{END_SUBJECT}.csv"
    if TEST_MODE:
        csv_name = csv_name.replace(".csv", "_TEST.csv")
    save_path = os.path.join(rootPath, csv_name)

    print(f"Starting batch processing (post-average-reference mode), Subjects: {START_SUBJECT} to {END_SUBJECT}")
    global_start_time = time.time()

    #Process and save incrementally to prevent data loss on interruption
    for i, sub_id in enumerate(valid_subjects, start=1):
        sub_start_time = time.time()
        res = evaluate_single_subject_parallel(sub_id, BS_LIST_TO_TEST)

        if res:
            df = pd.DataFrame(res)
            #Reorder columns for readability
            cols = ['Subject', 'Channel', 'Bs'] + [c for c in df.columns if c not in ['Subject', 'Channel', 'Bs']]
            df = df[cols]

            #Append to CSV per subject
            df.to_csv(
                save_path,
                mode=('a' if os.path.exists(save_path) else 'w'),
                header=(not os.path.exists(save_path)),
                index=False
            )

            sub_elapsed = (time.time() - sub_start_time) / 60
            total_elapsed = (time.time() - global_start_time) / 60
            print(f"[SUCCESS] Saved sub-{sub_id} ({len(res)} rows). [Progress: {i}/{len(valid_subjects)} | Total time: {total_elapsed:.1f}min | This sub: {sub_elapsed:.1f}min]")
        else:
            print(f"[ERROR] sub-{sub_id} produced no results.")

In [18]:
%%script false --no-raise-error
# ==========================================
# CELL 17: Baseline Batch Processing: FASTER + ASR
# ==========================================
!pip install asrpy hurst scikit-posthocs -q

import os
import gc
import mne
import numpy as np
import pandas as pd
import scipy as sp
import scipy.stats

#Configuration
START_SUBJECT = 1
END_SUBJECT = 30
datasetName = "N170"
rootPath = "/content/drive/MyDrive/EEG_Project/"
montageName = "standard_1020"

EEG_30 = ["FP1","F3","F7","FC3","C3","C5","P3","P7","P9","PO7","PO3","O1","Oz","Pz","CPz",
          "FP2","Fz","F4","F8","FC4","FCz","Cz","C4","C6","P4","P8","P10","PO8","PO4","O2"]
EOG_CH = ["HEOG_left","HEOG_right","VEOG_lower"]

usedChannels = EEG_30 + EOG_CH
montageChannelNames = EEG_30

#Baseline parameters
ASR_CONFIGS = [("offline", 13), ("offline", 20), ("online", 13), ("online", 20)]
WIN_SEC = 1.0
ONLINE_CALIB_WIN = 20

#Preprocessing functions for fair comparison
def fix_channel_types(raw):
    r = raw.copy()
    present = {c: 'eog' for c in EOG_CH if c in r.ch_names}
    if present:
        r.set_channel_types(present, verbose='ERROR')
    return r

def preprocess_avg_reref(raw):
    r = fix_channel_types(raw).load_data()
    r.pick(EEG_30)
    r.filter(1., None, verbose='ERROR')
    flat = mne.io.RawArray(
        np.zeros((1, r.n_times)),
        mne.create_info(['FLAT'], r.info['sfreq'], ['eeg']),
        verbose='ERROR'
    )
    r.add_channels([flat], force_update_info=True)
    r.set_eeg_reference('average', verbose='ERROR')
    r.drop_channels(['FLAT'])
    return r

#Single-subject baseline evaluation function
def run_baselines_for_subject(subject_id):
    try:
        raw_full = loadDataset(
            name=datasetName, subject=str(subject_id), montageName=montageName,
            usedChannels=usedChannels, rootPath=rootPath,
            montageChannelNames=montageChannelNames, verbose=False
        )
    except Exception as e:
        print(f"[WARNING] sub-{subject_id} loading failed, skipping: {e}")
        return []

    raw_full = fix_channel_types(raw_full).load_data()
    fs = raw_full.info['sfreq']

    try:
        events, event_id = mne.events_from_annotations(raw_full, verbose=False)
        codes = {v for k, v in event_id.items()
                 if str(k).strip().isdigit() and 1 <= int(str(k).strip()) <= 80}
        if codes:
            events = events[np.isin(events[:, 2], list(codes))]
        event_samples = events[:, 0].astype(int)
    except Exception:
        event_samples = np.array([], dtype=int)

    ref = preprocess_avg_reref(raw_full)
    ref_data = ref.get_data()
    eeg_names = ref.ch_names
    veog = raw_full.copy().pick(['VEOG_lower'])

    del raw_full
    gc.collect()

    evaluator = OnEEGWaveLAD_MetricsEvaluator(fs=fs, mode="real")
    rows = []

    #Strategy: Run each algorithm, evaluate immediately, then clear memory

    #(A) FASTER-inspired algorithm
    try:
        eeg_pre = ref.copy()
        clean_faster, n_bad = runICARemoveComponentsRunInverseICA(
            eegData=eeg_pre, baselineLenMs=0, ERPLenMs=0, sampleFreq=fs,
            numComponents=len(eeg_names), highPassTh=1, lowPassTh=int(fs//2),
            EOGData=veog, removeComponentsWithFASTERMethod=True, verbose=False
        )
        faster_out = clean_faster.copy().pick(eeg_names).get_data()

        #Evaluate FASTER immediately after completion
        L = min(ref_data.shape[1], faster_out.shape[1])
        for ci, ch in enumerate(eeg_names):
            m = evaluator.evaluate_channel(
                reference=ref_data[ci, :L].copy(),
                denoised=faster_out[ci, :L].copy(),
                raw=ref_data[ci, :L].copy(),
                event_samples=event_samples, mode="real"
            )
            rows.append({"Subject": f"sub-{subject_id}", "Channel": ch, "Method": "FASTER", **m})

        #Clear memory immediately after evaluation
        del eeg_pre, clean_faster, faster_out
        gc.collect()
        print(f"  [SUCCESS] FASTER done & memory cleared.")
    except Exception as e:
        print(f"  [ERROR] FASTER failed for sub-{subject_id}: {e}")

    #(B) Four ASR configurations
    for mode_, cut in ASR_CONFIGS:
        try:
            arr = ref_data.copy()
            if mode_ == "offline":
                Y = denoiseWithArtifaceSuspaceReconstructionwithASROffline(
                    arr, fs, arr.shape[0], cut, WIN_SEC, eeg_names, precleaned=True, verbose=False
                )
            else:
                Y = denoiseWithArtifaceSuspaceReconstructionwithASRRealTimeLogic(
                    arr, fs, int(ONLINE_CALIB_WIN * WIN_SEC * fs), arr.shape[0], cut, WIN_SEC,
                    eeg_names, precleaned=True, verbose=False
                )
            asr_out = np.asarray(Y)[:len(eeg_names)]

            #Evaluate ASR immediately after completion
            L = min(ref_data.shape[1], asr_out.shape[1])
            for ci, ch in enumerate(eeg_names):
                m = evaluator.evaluate_channel(
                    reference=ref_data[ci, :L].copy(),
                    denoised=asr_out[ci, :L].copy(),
                    raw=ref_data[ci, :L].copy(),
                    event_samples=event_samples, mode="real"
                )
                rows.append({"Subject": f"sub-{subject_id}", "Channel": ch, "Method": f"ASR_{mode_}_{cut}", **m})

            #Clear memory immediately after evaluation
            del arr, Y, asr_out
            gc.collect()
            print(f"  [SUCCESS] ASR {mode_}_{cut} done & memory cleared.")
        except Exception as e:
            print(f"  [ERROR] ASR {mode_}_{cut} failed for sub-{subject_id}: {e}")

    del ref, ref_data, veog, evaluator
    gc.collect()

    return rows

#Main loop
if __name__ == "__main__":
    base_path = os.path.join(rootPath, "onEEGWaveLAD_BASELINES.csv")
    valid_subjects = [str(i) for i in range(START_SUBJECT, END_SUBJECT + 1)]

    print(f"Starting baseline batch processing, Subjects: {START_SUBJECT} to {END_SUBJECT}")

    for subj in valid_subjects:
        print(f"\n{'='*40}\n=== Processing baselines for sub-{subj} ===\n{'='*40}")
        rows = run_baselines_for_subject(subj)

        if rows:
            df = pd.DataFrame(rows)
            df.to_csv(
                base_path,
                mode=('a' if os.path.exists(base_path) else 'w'),
                header=(not os.path.exists(base_path)),
                index=False
            )
            print(f"[SUCCESS] Saved sub-{subj} ({len(rows)} rows) to CSV.")

            del rows, df
            gc.collect()
        else:
            print(f"[ERROR] sub-{subj} produced no results.")

In [19]:
# ==========================================
# CELL 18: Statistical Analysis: Friedman + Post-hoc Tests
# ==========================================
import os
import itertools
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import friedmanchisquare, wilcoxon, rankdata

#Studentized Range Statistic q_alpha, alpha=0.05, df=inf
#Nemenyi uses q_alpha / sqrt(2) for the two-tailed all-pairs comparison.
Q005_INF = {2: 1.960, 3: 2.344, 4: 2.569, 5: 2.728, 6: 2.850, 7: 2.949,
            8: 3.031, 9: 3.102, 10: 3.164, 11: 3.219, 12: 3.268}

ALPHA = 0.05

#Metric orientation: "lower = better" metrics are negated before ranking so that rank 1 = best
METRICS_LOWER_BETTER = [
    "PSNR-Diff", "RMSE",
    "BPPE_Delta", "BPPE_Theta", "BPPE_Alpha", "BPPE_Beta", "BPPE_Gamma",
    "MSC_Delta", "MSC_Theta", "MSC_Gamma",
]
METRICS_HIGHER_BETTER = ["SNR-Diff", "Corr", "MI", "MSC_Alpha", "MSC_Beta"]
ALL_METRICS = METRICS_LOWER_BETTER + METRICS_HIGHER_BETTER


def _oriented(df, metric):
    """Return the metric column oriented so that larger == better."""
    v = df[metric].astype(float)
    return -v if metric in METRICS_LOWER_BETTER else v


def _nemenyi_cd(k, n, alpha=ALPHA):
    """CD = q_alpha * sqrt(k(k+1) / (6N))."""
    if k not in Q005_INF:
        raise ValueError(f"No tabulated q_alpha for k={k}")
    q = Q005_INF[k] / np.sqrt(2.0)
    return q * np.sqrt(k * (k + 1.0) / (6.0 * n))

def build_rank_matrix(df, methods, metrics, level="channel-specific",
                      method_col="Method"):
    """
    Build the (N x k) rank matrix passed to the Friedman test.

    Ranking is always done within one (Subject, Channel, metric) cell across
    the k methods, so each cell is a genuine repeated measurement. Only then are
    ranks aggregated.

    level:
      "channel-specific"   -> N = n_subject * n_channel
      "channel-aggregated" -> N = n_subject
      "metric-aggregated"  -> N = n_channel, per subject
    """
    k = len(methods)
    long_rows = []
    for metric in metrics:
        sub = df[[method_col, "Subject", "Channel", metric]].copy()
        sub["val"] = _oriented(sub, metric)
        wide = sub.pivot_table(index=["Subject", "Channel"],
                               columns=method_col, values="val")
        wide = wide.reindex(columns=methods).dropna()
        if wide.empty:
            continue
        #rank 1 = best; negate because rankdata ranks ascending
        r = np.apply_along_axis(lambda row: rankdata(-row), 1, wide.to_numpy())
        rdf = pd.DataFrame(r, index=wide.index, columns=methods)
        rdf["metric"] = metric
        long_rows.append(rdf.reset_index())

    if not long_rows:
        raise ValueError("No usable (Subject, Channel, metric) cells found.")
    long = pd.concat(long_rows, ignore_index=True)

    if level == "channel-specific":
        out = long.groupby(["Subject", "Channel"])[methods].mean()
        return out.to_numpy(), out.index
    if level == "channel-aggregated":
        per_ch = long.groupby(["Subject", "Channel"])[methods].mean()
        out = per_ch.groupby("Subject").mean()
        return out.to_numpy(), out.index
    if level == "metric-aggregated":
        return long
    raise ValueError(f"Unknown level: {level}")

def plot_cd_diagram(avg_ranks, names, cd, title, out_pdf=None, ax=None):
    """Demsar-style critical difference diagram."""
    order = np.argsort(avg_ranks)
    ranks = np.asarray(avg_ranks)[order]
    labels = [names[i] for i in order]
    k = len(ranks)

    created = ax is None
    if created:
        fig, ax = plt.subplots(figsize=(9, 2.2 + 0.34 * k))

    lo = float(np.floor(ranks.min() - 0.4))
    hi = float(np.ceil(ranks.max() + 0.4))
    ax.set_xlim(lo, hi)
    ax.set_ylim(0, 1)
    ax.axis("off")

    #top axis
    ax.hlines(0.86, lo, hi, color="k", lw=1.2)
    for t in np.arange(lo, hi + 1e-9, 0.5):
        ax.vlines(t, 0.86, 0.89, color="k", lw=1)
        ax.text(t, 0.92, f"{t:g}", ha="center", va="bottom", fontsize=8)

    #method markers, best (lowest rank) on the left
    half = int(np.ceil(k / 2))
    for i, (r, lab) in enumerate(zip(ranks, labels)):
        left = i < half
        row = i if left else k - 1 - i
        y = 0.72 - 0.085 * row
        xt = lo + 0.08 if left else hi - 0.08
        ax.plot([r, r], [0.86, y], color="0.4", lw=1)
        ax.plot([r, xt], [y, y], color="0.4", lw=1)
        ax.plot([r], [0.86], "o", color="crimson", ms=5, zorder=3)
        ax.text(xt, y, f" {lab} ({r:.2f})" if left else f"{lab} ({r:.2f}) ",
                ha="left" if left else "right", va="center", fontsize=9)

    #CD bar
    ax.hlines(0.97, lo + 0.05, lo + 0.05 + cd, color="k", lw=2.5)
    ax.vlines([lo + 0.05, lo + 0.05 + cd], 0.955, 0.985, color="k", lw=2.5)
    ax.text(lo + 0.05 + cd / 2, 0.995, f"CD = {cd:.3f}",
            ha="center", va="bottom", fontsize=9)

    #cliques: maximal groups whose rank span <= CD
    cliques = []
    for i in range(k):
        j = i
        while j + 1 < k and ranks[j + 1] - ranks[i] <= cd:
            j += 1
        if j > i:
            cliques.append((i, j))
    cliques = [c for c in cliques
               if not any(c != o and o[0] <= c[0] and c[1] <= o[1] for o in cliques)]
    for n, (i, j) in enumerate(cliques):
        y = 0.80 - 0.030 * n
        ax.hlines(y, ranks[i] - 0.03, ranks[j] + 0.03, color="steelblue", lw=3.5)

    ax.set_title(title, fontsize=10, pad=16)
    if created:
        plt.tight_layout()
        if out_pdf:
            plt.savefig(out_pdf, bbox_inches="tight")
        plt.close()
    return [(labels[i], labels[j]) for i, j in cliques]

def friedman_full_analysis(df, methods, metrics=None, method_col="Method",
                           out_dir=".", tag="baseline", make_per_metric=True):
    """Run the three Friedman levels plus per-metric CD diagrams."""
    if metrics is None:
        metrics = [m for m in ALL_METRICS if m in df.columns]
    methods = [m for m in methods if m in set(df[method_col].unique())]
    k = len(methods)
    os.makedirs(out_dir, exist_ok=True)
    print(f"Methods (k={k}): {methods}")
    print(f"Metrics ({len(metrics)}): {metrics}\n")
    results = {}

    #Levels 2 and 3: multi-subject Friedman + Nemenyi
    for level, shape_note in [("channel-specific", "N = subjects x channels"),
                              ("channel-aggregated", "N = subjects")]:
        R, idx = build_rank_matrix(df, methods, metrics, level, method_col)
        N = R.shape[0]
        chi2, p = friedmanchisquare(*[R[:, j] for j in range(k)])
        avg = R.mean(axis=0)
        cd = _nemenyi_cd(k, N)

        print(f"=== {level}  ({shape_note}) ===")
        print(f"  Rank matrix shape : {R.shape}")
        print(f"  Friedman chi2({k-1}) = {chi2:.2f}, p = {p:.3e}"
              f"  -> {'reject H0' if p < ALPHA else 'H0 retained'}")
        print(f"  Nemenyi CD (alpha={ALPHA}) = {cd:.4f}")
        for m, a in sorted(zip(methods, avg), key=lambda t: t[1]):
            print(f"    {m:24} avg rank = {a:.3f}")

        pdf = os.path.join(out_dir, f"CD_{tag}_{level}.pdf")
        cl = plot_cd_diagram(avg, methods, cd,
                             f"{level}: Friedman p={p:.2e}, N={N}, k={k}", pdf)
        print(f"  Non-significant cliques: {cl if cl else 'none (all differ)'}")
        print(f"  Saved: {pdf}\n")
        results[level] = {"chi2": chi2, "p": p, "avg_ranks": dict(zip(methods, avg)),
                          "CD": cd, "N": N}

    #Level 1: per-subject Friedman + Wilcoxon/Holm
    long = build_rank_matrix(df, methods, metrics, "metric-aggregated", method_col)
    per_subj, sig_counter = [], {c: 0 for c in itertools.combinations(methods, 2)}
    for subj, g in long.groupby("Subject"):
        Rs = g.groupby("Channel")[methods].mean().to_numpy()
        if Rs.shape[0] < 3:
            continue
        chi2, p = friedmanchisquare(*[Rs[:, j] for j in range(k)])
        per_subj.append({"Subject": subj, "N_channels": Rs.shape[0],
                         "chi2": chi2, "p": p})
        if p >= ALPHA:
            continue
        pairs, praw = [], []
        for a, b in itertools.combinations(range(k), 2):
            x, y = Rs[:, a], Rs[:, b]
            if np.allclose(x, y):
                continue
            pairs.append((methods[a], methods[b]))
            praw.append(wilcoxon(x, y, zero_method="wilcox").pvalue)
        #Holm-Bonferroni correction
        for pr, pair in zip(_holm(praw), pairs):
            if pr < ALPHA:
                sig_counter[pair] += 1

    ps = pd.DataFrame(per_subj)
    n_sig = int((ps["p"] < ALPHA).sum()) if not ps.empty else 0
    print("=== metric-aggregated / single-subject (30 x 7, repeated per subject) ===")
    print(f"  Subjects tested          : {len(ps)}")
    print(f"  Significant Friedman     : {n_sig}/{len(ps)}")
    print("  Pairs most often significant after Holm correction:")
    for pair, c in sorted(sig_counter.items(), key=lambda t: -t[1])[:8]:
        if c:
            print(f"    {pair[0]:22} vs {pair[1]:22} {c:3d}/{len(ps)} subjects")
    results["per_subject"] = ps
    results["wilcoxon_holm_counts"] = sig_counter
    print()

    #per-metric CD diagrams
    if make_per_metric:
        print("=== Per-metric CD diagrams (channel-specific level) ===")
        per_metric = {}
        for metric in metrics:
            try:
                R, _ = build_rank_matrix(df, methods, [metric],
                                         "channel-specific", method_col)
            except ValueError:
                print(f"  {metric:14} skipped (no usable cells)")
                continue
            N = R.shape[0]
            chi2, p = friedmanchisquare(*[R[:, j] for j in range(k)])
            avg, cd = R.mean(axis=0), _nemenyi_cd(k, N)
            best = methods[int(np.argmin(avg))]
            pdf = os.path.join(out_dir, f"CD_{tag}_metric_{metric}.pdf")
            plot_cd_diagram(avg, methods, cd,
                            f"{metric}: Friedman p={p:.2e}, N={N}", pdf)
            print(f"  {metric:14} p={p:.2e}  CD={cd:.3f}  best={best} "
                  f"(rank {avg.min():.2f})")
            per_metric[metric] = {"p": p, "avg_ranks": dict(zip(methods, avg)),
                                  "CD": cd, "best": best}
        results["per_metric"] = per_metric
    return results

def metric_aggregated_report(df, methods, metrics=None, method_col="Method",
                             out_dir=".", tag="Bs", max_panels=30):
    """
    Level 1: one Friedman test per subject, each on a 30 x 7 matrix.
    Collects every run for panel grid, cross-subject summary, and pooled diagram.
    """
    if metrics is None:
        metrics = [m for m in ALL_METRICS if m in df.columns]
    methods = [m for m in methods if m in set(df[method_col].unique())]
    k = len(methods)
    os.makedirs(out_dir, exist_ok=True)

    long = build_rank_matrix(df, methods, metrics, "metric-aggregated", method_col)

    rows, per_subj_ranks, panels = [], [], []
    sig = {c: 0 for c in itertools.combinations(methods, 2)}
    tested = 0
    for subj, g in long.groupby("Subject"):
        Rs = g.groupby("Channel")[methods].mean()
        if Rs.shape[0] < 3:
            continue
        A = Rs.to_numpy()
        chi2, p = friedmanchisquare(*[A[:, j] for j in range(k)])
        avg = A.mean(axis=0)
        cd = _nemenyi_cd(k, A.shape[0])
        tested += 1
        rows.append({"Subject": subj, "N_channels": A.shape[0], "chi2": chi2,
                     "p": p, "significant": p < ALPHA,
                     "best": methods[int(np.argmin(avg))], "CD": cd})
        per_subj_ranks.append(pd.Series(avg, index=methods, name=subj))
        panels.append((subj, avg, cd, p))

        if p < ALPHA:
            pairs, praw = [], []
            for a, b in itertools.combinations(range(k), 2):
                x, y = A[:, a], A[:, b]
                if np.allclose(x, y):
                    continue
                pairs.append((methods[a], methods[b]))
                praw.append(wilcoxon(x, y, zero_method="wilcox").pvalue)
            for pr, pair in zip(_holm(praw), pairs):
                if pr < ALPHA:
                    sig[pair] += 1

    summary = pd.DataFrame(rows)
    RK = pd.DataFrame(per_subj_ranks)
    return summary, RK, panels, sig, tested, methods, k

def _holm(pvals):
    """Holm-Bonferroni step-down adjusted p-values."""
    p = np.asarray(pvals, dtype=float)
    m = p.size
    if m == 0:
        return p
    order = np.argsort(p)
    adj = np.empty(m)
    run = 0.0
    for i, o in enumerate(order):
        run = max(run, (m - i) * p[o])
        adj[o] = min(run, 1.0)
    return adj

def _draw_panels(panels, methods, out_pdf, max_panels=30, ncol=3):
    """Grid of per-subject CD diagrams; 'n.s.' marks a non-significant Friedman."""
    sel = panels[:max_panels]
    nrow = int(np.ceil(len(sel) / ncol))
    fig, axes = plt.subplots(nrow, ncol, figsize=(6.0 * ncol, 2.5 * nrow))
    axes = np.atleast_1d(axes).ravel()
    for ax, (subj, avg, cd, p) in zip(axes, sel):
        star = "*" if p < ALPHA else "n.s."
        plot_cd_diagram(avg, methods, cd, f"{subj}  p={p:.1e} {star}",
                        ax=ax)
    for ax in axes[len(sel):]:
        ax.axis("off")
    plt.tight_layout()
    plt.savefig(out_pdf, bbox_inches="tight")
    plt.close()

def _draw_summary(summary, RK, sig, tested, methods, k, out_pdf):
    """Three-panel cross-subject view: rank spread, win counts, post-hoc heat map."""
    order = RK.mean(axis=0).sort_values().index.tolist()
    fig = plt.figure(figsize=(15, 4.4))
    gs = fig.add_gridspec(1, 3, width_ratios=[1.25, 1.0, 1.15], wspace=0.32)

    #(a) distribution of each method's mean rank over the 30 subjects
    ax = fig.add_subplot(gs[0, 0])
    data = [RK[m].values for m in order]
    bp = ax.boxplot(data, tick_labels=order, showmeans=True, widths=0.6,
                    patch_artist=True)
    for b in bp["boxes"]:
        b.set_facecolor("#cfe0f0")
        b.set_edgecolor("0.3")
    jit = np.random.default_rng(0)
    for i, m in enumerate(order, start=1):
        ax.plot(jit.normal(i, 0.055, len(RK)), RK[m].values,
                ".", color="0.45", ms=3, alpha=0.65, zorder=3)
    ax.set_ylabel("Mean rank within subject (1 = best)")
    ax.set_title("(a) Per-subject mean ranks", fontsize=10)
    ax.tick_params(axis="x", rotation=45, labelsize=8)
    ax.grid(axis="y", ls=":", alpha=0.5)

    #(b) how often each method is the within-subject winner
    ax = fig.add_subplot(gs[0, 1])
    wins = summary["best"].value_counts().reindex(order).fillna(0)
    ax.barh(range(len(order)), wins.values, color="crimson", alpha=0.8)
    ax.set_yticks(range(len(order)))
    ax.set_yticklabels(order, fontsize=8)
    ax.invert_yaxis()
    ax.set_xlabel(f"Subjects ranked best (out of {tested})")
    ax.set_title("(b) Within-subject wins", fontsize=10)
    for i, v in enumerate(wins.values):
        if v:
            ax.text(v, i, f" {int(v)}", va="center", fontsize=8)
    ax.grid(axis="x", ls=":", alpha=0.5)

    #(c) fraction of subjects where a pair survives Wilcoxon + Holm
    ax = fig.add_subplot(gs[0, 2])
    M = np.full((k, k), np.nan)
    for (a, b), c in sig.items():
        i, j = order.index(a), order.index(b)
        M[i, j] = M[j, i] = c / max(tested, 1)
    im = ax.imshow(M, cmap="YlOrRd", vmin=0, vmax=1)
    ax.set_xticks(range(k)); ax.set_xticklabels(order, rotation=45,
                                                ha="right", fontsize=7)
    ax.set_yticks(range(k)); ax.set_yticklabels(order, fontsize=7)
    for i in range(k):
        for j in range(k):
            if np.isfinite(M[i, j]):
                ax.text(j, i, f"{M[i, j]:.2f}", ha="center", va="center",
                        fontsize=6.5,
                        color="white" if M[i, j] > 0.6 else "0.15")
    ax.set_title("(c) Fraction of subjects with\nsignificant pair (Wilcoxon+Holm)",
                 fontsize=9)
    plt.colorbar(im, ax=ax, fraction=0.046)

    plt.savefig(out_pdf, bbox_inches="tight")
    plt.close()
    return order

def metric_aggregated_figures(df, methods, metrics=None, method_col="Method",
                              out_dir=".", tag="Bs", max_panels=30):
    summary, RK, panels, sig, tested, methods, k = metric_aggregated_report(
        df, methods, metrics, method_col, out_dir, tag, max_panels)

    p_panels = os.path.join(out_dir, f"CD_{tag}_metric-aggregated_panels.pdf")
    p_summ   = os.path.join(out_dir, f"CD_{tag}_metric-aggregated_summary.pdf")
    p_pool   = os.path.join(out_dir, f"CD_{tag}_metric-aggregated_pooled.pdf")

    _draw_panels(panels, methods, p_panels, max_panels)
    order = _draw_summary(summary, RK, sig, tested, methods, k, p_summ)

    #pooled: Friedman across subjects on the per-subject mean ranks
    A = RK[methods].to_numpy()
    chi2, p = friedmanchisquare(*[A[:, j] for j in range(k)])
    avg = A.mean(axis=0)
    cd = _nemenyi_cd(k, A.shape[0])
    cl = plot_cd_diagram(avg, methods, cd,
                         f"metric-aggregated, pooled over subjects: "
                         f"Friedman p={p:.2e}, N={A.shape[0]}, k={k}", p_pool)

    n_sig = int(summary["significant"].sum())
    print("=== metric-aggregated / single-subject ===")
    print(f"  Subjects tested        : {tested}   (matrix {RK.shape[0]}x{k}, "
          f"each subject {summary['N_channels'].iloc[0]}x{k})")
    print(f"  Significant Friedman   : {n_sig}/{tested}")
    print(f"  Pooled Friedman        : chi2={chi2:.2f}, p={p:.3e}, CD={cd:.4f}")
    print(f"  Rank order (best first): {order}")
    print(f"  Pooled cliques         : {cl if cl else 'none (all differ)'}")
    print("  Within-subject wins    :")
    for m, c in summary["best"].value_counts().items():
        print(f"    {m:24} {c:3d}/{tested}")
    print(f"  Saved: {p_panels}")
    print(f"  Saved: {p_summ}")
    print(f"  Saved: {p_pool}")
    return {"summary": summary, "rank_matrix": RK, "pooled": {
        "chi2": chi2, "p": p, "CD": cd, "avg_ranks": dict(zip(methods, avg))},
        "wilcoxon_holm_counts": sig}

#Main execution
try:
    from google.colab import drive
    drive.mount("/content/drive")
except ImportError:
    pass

ROOT_PATH = "/content/drive/MyDrive/EEG_Project/"
OUT_DIR = os.path.join(ROOT_PATH, "friedman_cd")

bs_csv = os.path.join(ROOT_PATH, "onEEGWaveLAD_Metrics_AVGREF_MASTER_30_Subjects.csv")
base_csv = os.path.join(ROOT_PATH, "onEEGWaveLAD_BASELINES.csv")

#Analysis A: the 7 Bs values as the k methods
print("#" * 70)
print("# Analysis A: 7 Bs values compared against each other")
print("#" * 70)
df_bs = pd.read_csv(bs_csv)
df_bs["Method"] = "Bs=" + df_bs["Bs"].astype(int).astype(str)
bs_methods = [f"Bs={b}" for b in sorted(df_bs["Bs"].unique())]

res_bs = friedman_full_analysis(df_bs, bs_methods, out_dir=OUT_DIR, tag="Bs")
res_ma_bs = metric_aggregated_figures(df_bs, bs_methods, out_dir=OUT_DIR, tag="Bs")

#Analysis B: onEEGWaveLAD vs the 5 baselines
print("\n" + "#" * 70)
print("# Analysis B: onEEGWaveLAD vs baselines")
print("#" * 70)
if os.path.exists(base_csv):
    df_b = pd.read_csv(base_csv)
    print("Methods present in baseline CSV:", sorted(df_b["Method"].unique()))

    #Include all 7 Bs values as representative data for merging, with distinguishing prefix
    rep = df_bs.copy()
    rep["Method"] = "onEEGWaveLAD-" + rep["Method"]
    print(f"Representative instantiations included: {sorted(rep['Method'].unique())}")
    df_all = pd.concat([df_b, rep], ignore_index=True)

    friedman_full_analysis(df_all, sorted(df_all["Method"].unique()),
                           out_dir=OUT_DIR, tag="baseline")
    res_ma_base = metric_aggregated_figures(df_all,
                                            sorted(df_all["Method"].unique()),
                                            out_dir=OUT_DIR, tag="baseline")
else:
    print(f"Baseline CSV not found: {base_csv}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
######################################################################
# Analysis A: 7 Bs values compared against each other
######################################################################
Methods (k=7): ['Bs=1', 'Bs=2', 'Bs=4', 'Bs=8', 'Bs=16', 'Bs=32', 'Bs=64']
Metrics (15): ['PSNR-Diff', 'RMSE', 'BPPE_Delta', 'BPPE_Theta', 'BPPE_Alpha', 'BPPE_Beta', 'BPPE_Gamma', 'MSC_Delta', 'MSC_Theta', 'MSC_Gamma', 'SNR-Diff', 'Corr', 'MI', 'MSC_Alpha', 'MSC_Beta']

=== channel-specific  (N = subjects x channels) ===
  Rank matrix shape : (900, 7)
  Friedman chi2(6) = 4621.85, p = 0.000e+00  -> reject H0
  Nemenyi CD (alpha=0.05) = 0.2124
    Bs=64                    avg rank = 3.098
    Bs=32                    avg rank = 3.315
    Bs=16                    avg rank = 3.634
    Bs=8                     avg rank = 3.965
    Bs=4                     avg rank = 4.31

In [20]:
# ==========================================
# CELL 19: Lower Bound Analysis: Diminishing Returns
# ==========================================
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter

METRICS_LOWER_BETTER = ['PSNR-Diff', 'RMSE', 'BPPE_Delta', 'BPPE_Theta', 'BPPE_Alpha',
                        'BPPE_Beta', 'BPPE_Gamma', 'MSC_Delta', 'MSC_Theta', 'MSC_Gamma']
METRICS_HIGHER_BETTER = ['SNR-Diff', 'Corr', 'MI', 'MSC_Alpha', 'MSC_Beta']
ALL_METRICS = METRICS_LOWER_BETTER + METRICS_HIGHER_BETTER

FAMILY_ARTEFACT = ['SNR-Diff', 'PSNR-Diff']
FAMILY_SIMILARITY = [m for m in ALL_METRICS if m not in FAMILY_ARTEFACT]


def _orient(v, m):
    return -v if m in METRICS_LOWER_BETTER else v


def _subject_matrix(df, m, bs_list):
    piv = df.pivot_table(index='Subject', columns='Bs', values=m, aggfunc='mean')
    return _orient(piv.reindex(columns=bs_list).values.astype(float), m)


def _effect_curve(df, m, bs_list):
    """Gain relative to the smallest Bs, in between-subject SD units."""
    V = _subject_matrix(df, m, bs_list)
    mean_c = np.nanmean(V, axis=0)
    sd = np.nanstd(V, ddof=1)
    if not np.isfinite(sd) or sd < 1e-12:
        return None
    return (mean_c - mean_c[0]) / sd


def _knee_on_curve(g, bs_list, scale='log2'):
    """Kneedle on defined scale (log2 or linear): max vertical distance above chord."""
    if scale == 'log2':
        x = np.log2(np.asarray(bs_list, dtype=float))
    else:
        #Linear axis X mapping
        x = np.asarray(bs_list, dtype=float)

    xn = (x - x.min()) / (x.max() - x.min())
    chord = g[0] + (g[-1] - g[0]) * xn
    dev = g - chord
    if not np.any(np.isfinite(dev)):
        return None, xn, chord
    return int(np.nanargmax(dev)), xn, chord


def knee_lower_bound_v2(csv_path, bs_list=None, families=None,
                        out_pdf="lower_bound_kneepoint_v2.pdf", scale='log2'):
    #Parameter: scale='log2' or 'linear'
    df = pd.read_csv(csv_path)
    if bs_list is None:
        bs_list = sorted(df['Bs'].unique())
    metrics = [m for m in ALL_METRICS if m in df.columns]
    print(f"Data: {df['Subject'].nunique()} Subjects x {df['Channel'].nunique()} "
          f"Channels x {len(bs_list)} Bs = {bs_list}\n")

    #per-metric effect curves and knees
    curves, knees = {}, {}
    for m in metrics:
        g = _effect_curve(df, m, bs_list)
        if g is None:
            continue
        curves[m] = g
        k, _, _ = _knee_on_curve(g, bs_list, scale=scale)
        if k is not None:
            knees[m] = bs_list[k]

    print(f"=== Per-metric total gain (SD units) and knee (Scale: {scale}) ===")
    print("  metric          gain(Bs1->max)   direction        knee Bs")
    for m in metrics:
        if m not in curves:
            continue
        tot = curves[m][-1]
        direction = "prefers large Bs" if tot > 0 else "prefers small Bs"
        print(f"  {m:14}  {tot:+13.2f}   {direction:16} {knees.get(m, '-')}")
    vote = Counter(knees.values())
    print(f"\n  Vote distribution: {dict(sorted(vote.items()))}")
    knee_vote = vote.most_common(1)[0][0] if vote else None
    print(f"  >> Majority metric knee (candidate X) = Bs {knee_vote}\n")

    #mean effect curve (no endpoint pinning)
    G = np.array([curves[m] for m in metrics if m in curves])
    mean_g = np.nanmean(G, axis=0)
    se_g = np.nanstd(G, axis=0, ddof=1) / np.sqrt(G.shape[0])
    k_mean, xn, chord = _knee_on_curve(mean_g, bs_list, scale=scale)

    print("=== Mean effect curve across all metrics (primary, unpinned) ===")
    print(f"  Bs             : {bs_list}")
    print(f"  Mean gain (SD) : {[f'{v:+.3f}' for v in mean_g]}")
    print(f"  >> Knee (Lower Bound X) = Bs {bs_list[k_mean]}")
    if 0 < k_mean < len(bs_list) - 1:
        sb = (mean_g[k_mean] - mean_g[0]) / (xn[k_mean] - xn[0] + 1e-9)
        sa = (mean_g[-1] - mean_g[k_mean]) / (xn[-1] - xn[k_mean] + 1e-9)
        print(f"  Slope before = {sb:+.3f} / after = {sa:+.3f} "
              f"-> ratio = {abs(sb) / (abs(sa) + 1e-9):.1f}x")
    print("  Sanity check: with min-max normalisation the endpoints would be forced")
    n_small = sum(1 for m in metrics if m in curves and curves[m][-1] < 0)
    if metrics:
        print(f"    to {n_small}/{len(curves)} = {n_small/max(len(curves),1):.3f} and "
              f"{1 - n_small/max(len(curves),1):.3f}; that artefact is gone here.\n")

    #family split
    fam = families if families is not None else {
        'Artefact-removal': FAMILY_ARTEFACT, 'Similarity': FAMILY_SIMILARITY}
    print("=== Per-family knee (each family is internally same-directional) ===")
    fam_out = {}
    for fname, mlist in fam.items():
        sel = [curves[m] for m in mlist if m in curves]
        if not sel:
            continue
        gf = np.nanmean(np.array(sel), axis=0)
        kf, _, _ = _knee_on_curve(gf, bs_list, scale=scale)
        fam_out[fname] = bs_list[kf] if kf is not None else None
        print(f"  {fname:18} ({len(sel):2d} metrics) knee = Bs {fam_out[fname]:<4} "
              f"total gain = {gf[-1]:+.2f} SD")
        print(f"    curve: {[f'{v:+.2f}' for v in gf]}")
    print()

    #Diagnostic: Identify and rank metrics by instability
    diag_stats = {}
    for m, c in curves.items():
        c = np.asarray(c, dtype=float)
        d = np.abs(np.diff(c))
        diag_stats[m] = {
            "range": np.nanmax(c) - np.nanmin(c),
            "max_step": np.nanmax(d),
            "tail_max_step": np.nanmax(d[len(d)//2:]),
            "n_sign_flips": int(np.sum(np.diff(np.sign(np.diff(c))) != 0)),
            "n_nan": int(np.isnan(c).sum())
        }
    rank = pd.DataFrame(diag_stats).T.sort_values("tail_max_step", ascending=False)
    print("=== Metric stability diagnostics (Ranked by tail instability) ===")
    print(rank.round(3))
    worst_offender = rank.index[0] if not rank.empty else None
    print(f"\n  >> Worst offender (most unstable in tail): {worst_offender}\n")

    #Plot
    fig, ax = plt.subplots(1, 2, figsize=(14.5, 5.5))

    cmap = plt.get_cmap('tab20')
    for i, m in enumerate(curves):
        ax[0].plot(xn, curves[m], color=cmap(i), lw=1.5, alpha=0.9, zorder=1, label=m)

    ax[0].plot(xn, mean_g, 'o-', color='black', lw=3.5, zorder=5, label='Mean gain (SD units)')
    ax[0].fill_between(xn, mean_g - 1.96 * se_g, mean_g + 1.96 * se_g,
                       color='black', alpha=.15, zorder=2, label='95% CI across metrics')
    ax[0].plot([xn[0], xn[-1]], [mean_g[0], mean_g[-1]], 'k--', lw=1.5, zorder=3, label='Start-end chord')

    ax[0].axvline(xn[k_mean], color='red', ls=':', lw=2.5, zorder=3,
                  label=f'Knee = Bs {bs_list[k_mean]}')

    ax[0].axhline(0, color='k', lw=.8, zorder=3)
    ax[0].set_xticks(xn); ax[0].set_xticklabels(bs_list)

    #Dynamic X-axis label adjustment
    x_label_str = 'Bs (log2 spacing)' if scale == 'log2' else 'Bs (linear spacing)'
    ax[0].set_xlabel(x_label_str); ax[0].set_ylabel('Gain vs Bs=1 (SD units)')
    ax[0].set_title(f'Lower Bound X v2 (Kneedle - {scale}): All Individual Metrics')
    ax[0].legend(fontsize=7, ncol=2, loc='best')

    for fname, mlist in fam.items():
        sel = [curves[m] for m in mlist if m in curves]
        if sel:
            ax[1].plot(xn, np.nanmean(np.array(sel), axis=0), 'o-', lw=2,
                       label=f'{fname} (n={len(sel)})')
    ax[1].axhline(0, color='k', lw=.8)
    ax[1].set_xticks(xn); ax[1].set_xticklabels(bs_list)
    ax[1].set_xlabel(x_label_str); ax[1].set_ylabel('Gain vs Bs=1 (SD units)')
    ax[1].set_title('Metric families read separately')
    ax[1].legend(fontsize=9)

    plt.tight_layout(); plt.savefig(out_pdf, bbox_inches='tight'); plt.close()
    print(f"Saved figure to: {out_pdf}")

    return {'X_knee': bs_list[k_mean], 'X_vote': knee_vote,
            'X_by_family': fam_out, 'mean_gain': mean_g}


if __name__ == "__main__":
    #Mount Google Drive
    try:
        from google.colab import drive
        drive.mount('/content/drive')
    except ImportError:
        pass

    ROOT_PATH = "/content/drive/MyDrive/EEG_Project/"

    #Stage 1: full coarse grid [1, 2, 4, 8, 16, 32, 64]
    csv_stage1 = os.path.join(ROOT_PATH, "onEEGWaveLAD_Metrics_AVGREF_MASTER_30_Subjects.csv")
    pdf_stage1 = os.path.join(ROOT_PATH, "lower_bound_stage1_v2.pdf")

    print("[Stage 1: Coarse Grid 1,2,4,8,16,32,64]")
    if os.path.exists(csv_stage1):
        #Stage 1: maintain log2 axis
        knee_lower_bound_v2(csv_stage1, out_pdf=pdf_stage1, scale='log2')
    else:
        print(f"File not found: {csv_stage1}")

    print("\n" + "=" * 60 + "\n")

    #Stage 2: refined grid 1-18
    csv_stage2 = os.path.join(ROOT_PATH, "onEEGWaveLAD_Metrics_STAGE2_REFINED.csv")
    pdf_stage2 = os.path.join(ROOT_PATH, "lower_bound_stage2_v2_refined.pdf")

    print("[Stage 2: Refined Grid 1-18]")
    if os.path.exists(csv_stage2):
        #Stage 2: enable linear axis
        knee_lower_bound_v2(csv_stage2, out_pdf=pdf_stage2, scale='linear')
    else:
        print(f"Stage 2 skipped: {os.path.basename(csv_stage2)} not found yet.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
[Stage 1: Coarse Grid 1,2,4,8,16,32,64]
Data: 30 Subjects x 30 Channels x 7 Bs = [np.int64(1), np.int64(2), np.int64(4), np.int64(8), np.int64(16), np.int64(32), np.int64(64)]

=== Per-metric total gain (SD units) and knee (Scale: log2) ===
  metric          gain(Bs1->max)   direction        knee Bs
  PSNR-Diff               -0.55   prefers small Bs 1
  RMSE                    +0.05   prefers large Bs 2
  BPPE_Delta              +1.81   prefers large Bs 4
  BPPE_Theta              +0.84   prefers large Bs 4
  BPPE_Alpha              +1.30   prefers large Bs 4
  BPPE_Beta               +1.25   prefers large Bs 4
  BPPE_Gamma              +0.34   prefers large Bs 4
  MSC_Delta               -2.59   prefers small Bs 1
  MSC_Theta               -2.72   prefers small Bs 1
  MSC_Gamma               -2.41   prefers small Bs 1
  SNR-Diff                -0.13   prefer

In [21]:
# ==========================================
# CELL 20: Convergence Bound Analysis
# ==========================================
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

#Metric polarities
METRICS_LOWER_BETTER = ['PSNR-Diff', 'RMSE', 'BPPE_Delta', 'BPPE_Theta', 'BPPE_Alpha',
                        'BPPE_Beta', 'BPPE_Gamma', 'MSC_Delta', 'MSC_Theta', 'MSC_Gamma']
METRICS_HIGHER_BETTER = ['SNR-Diff', 'Corr', 'MI', 'MSC_Alpha', 'MSC_Beta']
ALL_METRICS = METRICS_LOWER_BETTER + METRICS_HIGHER_BETTER

#Tentative family split: artefact-removal-rewarding vs similarity-to-original
FAMILY_ARTEFACT = ['SNR-Diff', 'PSNR-Diff']
FAMILY_SIMILARITY = [m for m in ALL_METRICS if m not in FAMILY_ARTEFACT]

MAJORITY = 0.70      #fraction of metrics that must have converged
Z_CRIT = 1.96        #two-sided 5%: "indistinguishable from zero"
HALF_NORMAL = np.sqrt(2.0 / np.pi)   #E|X|/sigma for X ~ N(0, sigma)


def _orient(v, m):
    """Apply polarity orientation so higher is always better."""
    return -v if m in METRICS_LOWER_BETTER else v


def _subject_matrix(df, m, bs_list):
    """(n_subj, n_bs) subject-mean curve (mean over channels), oriented."""
    piv = df.pivot_table(index='Subject', columns='Bs', values=m, aggfunc='mean')
    piv = piv.reindex(columns=bs_list)
    return _orient(piv.values.astype(float), m)


def _step_stats(V):
    """Paired per-step change across subjects: mean, SE, n_used."""
    d = np.diff(V, axis=1)                       #(n_subj, n_step)
    n = np.sum(~np.isnan(d), axis=0)
    with np.errstate(invalid='ignore'):
        mean_d = np.nanmean(d, axis=0)
        sd_d = np.nanstd(d, axis=0, ddof=1)
    se_d = sd_d / np.sqrt(np.maximum(n, 1))
    return mean_d, se_d, n


def _bound_from_flags(bs_list, flags, majority=MAJORITY):
    frac = flags.mean(axis=0)
    n_step = len(frac)
    for i in range(n_step):
        if all(frac[j] >= majority for j in range(i, n_step)):
            return bs_list[i], frac
    return None, frac


def analyze_convergence_v2(csv_path, bs_list=None, taus=(0.01, 0.05, 0.10),
                           families=None, out_pdf="convergence_bound_v2.pdf"):
    df = pd.read_csv(csv_path)
    if bs_list is None:
        bs_list = sorted(df['Bs'].unique())
    metrics = [m for m in ALL_METRICS if m in df.columns]
    steps = [f"{bs_list[i]}->{bs_list[i+1]}" for i in range(len(bs_list) - 1)]

    print(f"Data: {df['Subject'].nunique()} Subjects x {df['Channel'].nunique()} "
          f"Channels x {len(bs_list)} Bs = {bs_list}")
    print(f"Metrics available: {len(metrics)}/{len(ALL_METRICS)}\n")

    stats, sig_flags = {}, []
    abs_norm_mean, abs_norm_median = [], []

    #Gather per-metric step statistics (Parallel Pipelines)
    for m in metrics:
        V = _subject_matrix(df, m, bs_list)
        mean_d, se_d, n = _step_stats(V)
        stats[m] = (mean_d, se_d, n)

        with np.errstate(invalid='ignore', divide='ignore'):
            z = np.abs(mean_d) / np.where(se_d > 0, se_d, np.nan)
        sig_flags.append(np.nan_to_num(z, nan=0.0) < Z_CRIT)

        #Pipeline 1: Mean-based normalization
        rng_mean = np.nanmax(np.nanmean(V, axis=0)) - np.nanmin(np.nanmean(V, axis=0))
        abs_norm_mean.append(np.abs(mean_d) / rng_mean if rng_mean > 1e-12 else np.full(len(mean_d), np.nan))

        #Pipeline 2: Median-based computation directly across 30 subjects
        d = np.diff(V, axis=1)
        median_d = np.nanmedian(d, axis=0)
        rng_median = np.nanmax(np.nanmedian(V, axis=0)) - np.nanmin(np.nanmedian(V, axis=0))
        abs_norm_median.append(np.abs(median_d) / rng_median if rng_median > 1e-12 else np.full(len(median_d), np.nan))

    sig_flags = np.array(sig_flags)
    D_abs_mean = np.array(abs_norm_mean)
    D_abs_median = np.array(abs_norm_median)

    mean_abs_global = np.nanmean(D_abs_mean, axis=0)
    median_abs_global = np.nanmedian(D_abs_median, axis=0)

    #(1) absolute criterion, no tau
    print("=== Criterion A: change indistinguishable from zero (Mean-based stats) ===")
    frac_sig = sig_flags.mean(axis=0)
    for s, f in zip(steps, frac_sig):
        print(f"    {s:>9}: {f:.0%} converged")
    X_abs, _ = _bound_from_flags(bs_list, sig_flags)
    print(f"  >> Bound X (absolute) = {X_abs if X_abs else 'no step is noise-level; change persists to the largest Bs'}\n")

    #(2) noise floors (Mean vs Median)
    floors_mean, floors_median = [], []
    for m in metrics:
        mean_d, se_d, _ = stats[m]
        V = _subject_matrix(df, m, bs_list)

        rng_mean = np.nanmax(np.nanmean(V, axis=0)) - np.nanmin(np.nanmean(V, axis=0))
        floors_mean.append(se_d * HALF_NORMAL / rng_mean if rng_mean > 1e-12 else np.full(len(se_d), np.nan))

        rng_median = np.nanmax(np.nanmedian(V, axis=0)) - np.nanmin(np.nanmedian(V, axis=0))
        floors_median.append(se_d / rng_median if rng_median > 1e-12 else np.full(len(se_d), np.nan))

    floor_mean = np.nanmean(np.array(floors_mean), axis=0)
    floor_median = np.nanmedian(np.array(floors_median), axis=0)

    print("=== Criterion B: mean|delta| vs its own noise floor ===")
    print("       step   mean|d|   noise floor   verdict")
    for s, ma, fl in zip(steps, mean_abs_global, floor_mean):
        verdict = "within noise" if ma <= fl else "above noise"
        print(f"    {s:>9}   {ma:7.3f}   {fl:11.3f}   {verdict}")
    print()

    print("=== Robustness Check: True Median-based pipeline (Median across subjects) ===")
    print("       step   median|d|   noise floor")
    for s, med, fl in zip(steps, median_abs_global, floor_median):
        verdict = "within noise" if med <= fl else "above noise"
        print(f"    {s:>9}   {med:7.3f}   {fl:11.3f}   {verdict}")
    print()

    #(3) family split
    fam = families if families is not None else {
        'Artefact-removal': FAMILY_ARTEFACT, 'Similarity': FAMILY_SIMILARITY}
    print("=== Per metric family (Mean-based) ===")
    fam_out = {}
    for fname, mlist in fam.items():
        idx = [i for i, m in enumerate(metrics) if m in mlist]
        if not idx: continue
        Xf, _ = _bound_from_flags(bs_list, sig_flags[idx])
        fam_out[fname] = Xf
        curve = np.nanmean(D_abs_mean[idx], axis=0)
        print(f"    {fname:18} ({len(idx):2d} metrics) X = {str(Xf):>6}  mean|d| = {np.round(curve, 3)}")
    print()

    #Plot
    fig, ax = plt.subplots(2, 2, figsize=(14.5, 11))
    x = np.arange(len(steps))
    cmap = plt.get_cmap('tab20')

    #ROW 1: MEAN-BASED
    #[Top-Left] 15 Mean lines + Global Mean
    for i, m in enumerate(metrics):
        ax[0, 0].plot(x, D_abs_mean[i], color=cmap(i), lw=1.5, alpha=0.9, zorder=1, label=m)
    ax[0, 0].plot(x, mean_abs_global, 'o-', color='black', lw=3.5, zorder=5, label='Mean |Δ| across all metrics')
    ax[0, 0].plot(x, floor_mean, 'r--', lw=2, zorder=4, label='Noise Floor (E|mean(d)|)')

    if X_abs is not None and X_abs in bs_list:
        idx_val = bs_list.index(X_abs)
        if idx_val < len(steps):
            ax[0, 0].axvline(idx_val, color='red', ls=':', lw=2.5, zorder=3, label=f'Bound X = Bs {X_abs}')

    ax[0, 0].set_xticks(x); ax[0, 0].set_xticklabels(steps, rotation=45)
    ax[0, 0].set_xlabel('Adjacent Bs Steps'); ax[0, 0].set_ylabel('Normalized |Δ| (Step Change)')
    ax[0, 0].set_title('Convergence (Mean-based): Absolute Change per Step')
    ax[0, 0].legend(fontsize=7, ncol=2, loc='best')

    #[Top-Right] Mean Family split
    for fname, mlist in fam.items():
        idx = [i for i, m in enumerate(metrics) if m in mlist]
        if idx:
            ax[0, 1].plot(x, np.nanmean(D_abs_mean[idx], axis=0), 'o-', lw=2, label=f'{fname} Mean (n={len(idx)})')
    ax[0, 1].plot(x, floor_mean, 'r--', lw=1.5, label='Mean Noise Floor')
    ax[0, 1].set_xticks(x); ax[0, 1].set_xticklabels(steps, rotation=45)
    ax[0, 1].set_xlabel('Adjacent Bs Steps'); ax[0, 1].set_ylabel('Normalized |Δ| (Step Change)')
    ax[0, 1].set_title('Metric families read separately (Mean)')
    ax[0, 1].legend(fontsize=9)


    #ROW 2: MEDIAN-BASED
    #[Bottom-Left] 15 Median lines + Global Median
    for i, m in enumerate(metrics):
        ax[1, 0].plot(x, D_abs_median[i], color=cmap(i), lw=1.5, alpha=0.9, zorder=1)
    ax[1, 0].plot(x, median_abs_global, 's-', color='dodgerblue', lw=3.5, zorder=5, label='Median |Δ| across all metrics')
    ax[1, 0].plot(x, floor_median, 'r--', lw=2, zorder=4, label='Noise Floor (E|median(d)|)')

    if X_abs is not None and X_abs in bs_list:
        idx_val = bs_list.index(X_abs)
        if idx_val < len(steps):
            ax[1, 0].axvline(idx_val, color='red', ls=':', lw=2.5, zorder=3, label=f'Bound X (from Mean test)')

    ax[1, 0].set_xticks(x); ax[1, 0].set_xticklabels(steps, rotation=45)
    ax[1, 0].set_xlabel('Adjacent Bs Steps'); ax[1, 0].set_ylabel('Normalized |Δ| (Step Change)')
    ax[1, 0].set_title('Robustness Check (True Median-based pipeline)')
    ax[1, 0].legend(fontsize=9, loc='best')

    #[Bottom-Right] Median Family split
    for fname, mlist in fam.items():
        idx = [i for i, m in enumerate(metrics) if m in mlist]
        if idx:
            ax[1, 1].plot(x, np.nanmedian(D_abs_median[idx], axis=0), 's-', lw=2, label=f'{fname} Median (n={len(idx)})')
    ax[1, 1].plot(x, floor_median, 'r--', lw=1.5, label='Median Noise Floor')
    ax[1, 1].set_xticks(x); ax[1, 1].set_xticklabels(steps, rotation=45)
    ax[1, 1].set_xlabel('Adjacent Bs Steps'); ax[1, 1].set_ylabel('Normalized |Δ| (Step Change)')
    ax[1, 1].set_title('Metric families read separately (Median)')
    ax[1, 1].legend(fontsize=9)

    plt.tight_layout()
    plt.savefig(out_pdf, bbox_inches='tight')
    plt.close()

    print(f"Saved figure to: {out_pdf}")

    return {'X_abs': X_abs, 'X_by_family': fam_out, 'mean_abs': mean_abs_global, 'median_abs': median_abs_global}

if __name__ == "__main__":
    #Mount Google Drive
    try:
        from google.colab import drive
        drive.mount('/content/drive')
    except ImportError:
        pass

    ROOT_PATH = "/content/drive/MyDrive/EEG_Project/"

    csv_stage1 = os.path.join(ROOT_PATH, "onEEGWaveLAD_Metrics_AVGREF_MASTER_30_Subjects.csv")
    pdf_stage1 = os.path.join(ROOT_PATH, "convergence_stage1_v2.pdf")

    print("[Stage 1: Coarse Grid Convergence 1,2,4,8,16,32,64]")
    if os.path.exists(csv_stage1):
        analyze_convergence_v2(csv_stage1, out_pdf=pdf_stage1)
    else:
        print(f"File not found: {csv_stage1}")

    print("\n" + "=" * 60 + "\n")

    csv_stage2 = os.path.join(ROOT_PATH, "onEEGWaveLAD_Metrics_STAGE2_REFINED.csv")
    pdf_stage2 = os.path.join(ROOT_PATH, "convergence_stage2_v2_refined.pdf")

    print("[Stage 2: Refined Grid 6-18]")
    if os.path.exists(csv_stage2):
        analyze_convergence_v2(csv_stage2, out_pdf=pdf_stage2)
    else:
        print(f"Stage 2 skipped: {os.path.basename(csv_stage2)} not found yet.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
[Stage 1: Coarse Grid Convergence 1,2,4,8,16,32,64]
Data: 30 Subjects x 30 Channels x 7 Bs = [np.int64(1), np.int64(2), np.int64(4), np.int64(8), np.int64(16), np.int64(32), np.int64(64)]
Metrics available: 15/15

=== Criterion A: change indistinguishable from zero (Mean-based stats) ===
         1->2: 20% converged
         2->4: 13% converged
         4->8: 13% converged
        8->16: 13% converged
       16->32: 13% converged
       32->64: 0% converged
  >> Bound X (absolute) = no step is noise-level; change persists to the largest Bs

=== Criterion B: mean|delta| vs its own noise floor ===
       step   mean|d|   noise floor   verdict
         1->2     0.378         0.083   above noise
         2->4     0.194         0.055   above noise
         4->8     0.149         0.054   above noise
        8->16     0.098         0.012   above noise
       16->32 